# Phase 6 - Image-file Audit, Image Metadata Validation and Exploratory Image Analysis

This notebook continues the Hydrogel Bioink Data Curation Project from the verified Phase 5 checkpoint. It audits the original `images.zip` archive, validates the archive against `image_index.csv`, checks image integrity, extracts simple image descriptors, and creates a traceable image evidence layer for later computer-vision or machine-learning readiness assessment.

The scientific aim is deliberately conservative:

- inventory the original hydrogel printing images;
- compare actual image files against source metadata;
- describe image properties and simple computed features;
- preserve unresolved scientific boundaries;
- avoid treating computed image features as validated printability, morphology, cell viability, or clinical outcomes.

In polymer-informatics terms, this notebook is not yet a predictive modelling notebook. It is a data engineering and evidence-quality notebook. A reliable materials model depends on reliable units, provenance, observation hierarchy, image metadata, and sample-linkage assumptions. If those foundations are weak, the model may appear sophisticated while learning from noise, file names, or leakage.

## Reproducibility Contract

This notebook is designed so that after the correct files are in place, the workflow can be run as:

SAVE NOTEBOOK -> CLOSE JUPYTER -> REOPEN JUPYTER -> RESTART KERNEL -> RUN ALL CELLS -> COMPLETE SUCCESSFUL EXECUTION

It must not depend on variables left in memory, cells run out of order, hidden manual objects, or notebooks from earlier phases remaining open.

Expected local file location for the large image archive:

`data/raw/zenodo_19602891/images.zip`

Expected metadata file:

`data/raw/zenodo_19602891/image_index.csv`

The notebook also looks for Phase 4 and Phase 5 checkpoint files inside `data/processed/`, but those checkpoint files are treated as supporting context rather than raw experimental evidence.

In [3]:
# Phase 6 dependency check.
# This cell intentionally checks only packages that are required to run the notebook.
# scikit-image is optional and is NOT imported here because some local environments
# have NumPy 2.x with an older compiled scikit-image wheel, which raises a binary
# compatibility error before the real Phase 6 work can start.

import importlib
import os
import sys
from importlib import metadata
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib_cache"))

REQUIRED_MODULES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "PIL": "Pillow",
}

required_errors = []
for module_name, package_name in REQUIRED_MODULES.items():
    try:
        importlib.import_module(module_name)
    except Exception as exc:
        required_errors.append((module_name, package_name, type(exc).__name__, str(exc)))

if required_errors:
    install_list = " ".join(package for _, package, _, _ in required_errors)
    details = "\n".join(
        f"- {package} ({module}): {error_type}: {message}"
        for module, package, error_type, message in required_errors
    )
    raise ImportError(
        "One or more required Python packages could not be imported.\n"
        + details
        + "\n\nInstall or repair them in the active Jupyter environment, for example:\n"
        + f"    python -m pip install --upgrade {install_list}\n\n"
        + "After installation, restart the Jupyter kernel and run all cells again."
    )

print("Required packages are importable.")
for _, package_name in REQUIRED_MODULES.items():
    try:
        version = metadata.version(package_name)
    except Exception:
        version = "version not reported"
    print(f"- {package_name}: {version}")

print("Optional scikit-image: not required and intentionally not imported.")
print("Optional scikit-image testing can be enabled by launching Jupyter with PHASE6_USE_SKIMAGE=1 after the environment is repaired.")
print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])


Required packages are importable.
- pandas: 2.2.3
- numpy: 2.5.3
- matplotlib: 3.11.2
- Pillow: 12.3.0
Optional scikit-image: not required and intentionally not imported.
Optional scikit-image testing can be enabled by launching Jupyter with PHASE6_USE_SKIMAGE=1 after the environment is repaired.
Python executable: C:\Users\sufer\anaconda3\envs\hydrogel-bioink\python.exe
Python version: 3.12.15


In [4]:
# Core imports and global configuration.

from __future__ import annotations

import csv
import datetime as dt
import hashlib
import html
import json
import math
import os
import platform
import re
import textwrap
import time
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass
from io import BytesIO
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib_cache"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import HTML, Markdown, display
except ImportError:
    class Markdown(str):
        pass
    class HTML(str):
        pass
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)
from PIL import Image, ImageFile, UnidentifiedImageError

# NumPy-safe optional scikit-image handling.
# By default Phase 6 avoids importing scikit-image because older scikit-image wheels
# can be binary-incompatible with NumPy 2.x. The notebook provides a NumPy-only
# Otsu fallback, so scikit-image is not needed for a successful Run All.
SKIMAGE_REQUESTED = os.environ.get("PHASE6_USE_SKIMAGE", "0").strip().lower() in {"1", "true", "yes", "y"}
SKIMAGE_AVAILABLE = False
SKIMAGE_IMPORT_STATUS = "skipped"
SKIMAGE_IMPORT_MESSAGE = "scikit-image was skipped; NumPy-only image feature methods will be used."
threshold_otsu = None

if SKIMAGE_REQUESTED:
    try:
        from skimage.filters import threshold_otsu as _skimage_threshold_otsu
        threshold_otsu = _skimage_threshold_otsu
        SKIMAGE_AVAILABLE = True
        SKIMAGE_IMPORT_STATUS = "available"
        SKIMAGE_IMPORT_MESSAGE = "scikit-image import succeeded."
    except Exception as exc:
        threshold_otsu = None
        SKIMAGE_AVAILABLE = False
        SKIMAGE_IMPORT_STATUS = "unavailable"
        SKIMAGE_IMPORT_MESSAGE = f"scikit-image import failed and was safely skipped: {type(exc).__name__}: {exc}"

ImageFile.LOAD_TRUNCATED_IMAGES = False

RUN_STARTED_AT = dt.datetime.now(dt.timezone.utc)
RUN_ID = RUN_STARTED_AT.strftime("phase6_%Y%m%dT%H%M%SZ")

CONFIG = {
    "phase_name": "Phase 6 - Image-file Audit, Image Metadata Validation and Exploratory Image Analysis",
    "archive_name": "images.zip",
    "image_index_name": "image_index.csv",
    "raw_subdir": Path("data") / "raw" / "zenodo_19602891",
    "processed_subdir": Path("data") / "processed",
    "output_subdir": Path("data") / "processed" / "image_file_audit_optional",
    "figures_subdir_name": "figures",
    "max_preview_dimension_px": 768,
    "feature_dimension_px": 512,
    "example_grid_limit": 12,
    "processed_comparison_limit": 4,
    "compute_archive_sha256": False,
    "compute_image_sha256": True,
    "run_zip_integrity_test": True,
    "progress_every_n_images": 100,
    "random_seed": 42,
}

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 180,
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
})

print(CONFIG["phase_name"])
print("Run ID:", RUN_ID)
print("scikit-image optional status:", SKIMAGE_IMPORT_STATUS)
print("scikit-image note:", SKIMAGE_IMPORT_MESSAGE)


Phase 6 - Image-file Audit, Image Metadata Validation and Exploratory Image Analysis
Run ID: phase6_20261012T042010Z
scikit-image optional status: skipped
scikit-image note: scikit-image was skipped; NumPy-only image feature methods will be used.


## Data Science and Polymer Science Link

A printed hydrogel image is both a scientific observation and a data object.

From the polymer science side, the image may show a hydrogel bioink after extrusion, deposition, spreading, shape retention, and visible-light-assisted crosslinking. For phenol-modified systems such as ALG-Ph, HA-Ph, or ALG-Ph/HA-Ph blends, the visible print may be affected by formulation, polymer concentration, pH, enzymatic or photo-crosslinking conditions, storage modulus, loss modulus, yield-like behaviour, filament fusion, grid geometry, illumination, camera settings, background and sample handling.

From the data science side, the same image is a pixel array. Each pixel stores intensity or colour-channel values. When the notebook calculates brightness, contrast, edge density, foreground area or a simple visual hash, it is converting pixels into numeric descriptors. Those descriptors are useful because they make images searchable, comparable and auditable.

The key scientific boundary is this: a pixel descriptor is not automatically a materials property. A high foreground-area fraction may mean a larger visible printed region, but it may also reflect cropping, lighting, background colour or image scale. A low-contrast image may reflect a faint hydrogel, but it may also reflect camera exposure. Phase 6 therefore treats image features as **review cues**, not as final labels for printability, viability, mechanical strength or clinical performance.

This is exactly why image-file audit comes before modelling. Good materials informatics is not just about models; it is about knowing what each row, file and feature really means before asking an algorithm to learn from it.


In [6]:
# Project-root discovery and output-folder creation.


def _unique_existing_paths(paths):
    seen = set()
    out = []
    for p in paths:
        p = Path(p).expanduser().resolve()
        if p not in seen and p.exists():
            out.append(p)
            seen.add(p)
    return out


def find_project_root() -> Path:
    """Find the project root using an environment override or nearby data/raw folder."""
    override = os.environ.get("PHASE6_PROJECT_ROOT", "").strip()
    candidates = []
    if override:
        candidates.append(Path(override))
    cwd = Path.cwd()
    candidates.extend([cwd, *cwd.parents])

    for candidate in candidates:
        candidate = Path(candidate).expanduser().resolve()
        if (candidate / CONFIG["raw_subdir"]).exists():
            return candidate

    # If the notebook is opened inside a notebooks folder before data/raw exists,
    # use the parent as a sensible project root.
    if cwd.name.lower() in {"notebook", "notebooks"}:
        return cwd.parent.resolve()
    return cwd.resolve()


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / CONFIG["raw_subdir"]
PROCESSED_DIR = PROJECT_ROOT / CONFIG["processed_subdir"]
OUTPUT_DIR = PROJECT_ROOT / CONFIG["output_subdir"]
FIGURE_DIR = OUTPUT_DIR / CONFIG["figures_subdir_name"]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("Phase 6 output directory:", OUTPUT_DIR)
print("Phase 6 figure directory:", FIGURE_DIR)

Project root: C:\Users\sufer\Documents\hydrogel-bioink
Raw data directory: C:\Users\sufer\Documents\hydrogel-bioink\data\raw\zenodo_19602891
Processed data directory: C:\Users\sufer\Documents\hydrogel-bioink\data\processed
Phase 6 output directory: C:\Users\sufer\Documents\hydrogel-bioink\data\processed\image_file_audit_optional
Phase 6 figure directory: C:\Users\sufer\Documents\hydrogel-bioink\data\processed\image_file_audit_optional\figures


In [7]:
# Helper functions used throughout the notebook.

IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}


def normalise_text(value) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip().replace("\\", "/").lower()


def normalise_filename(value) -> str:
    value = normalise_text(value)
    return Path(value).name.lower() if value else ""


def normalise_stem(value) -> str:
    value = normalise_filename(value)
    return Path(value).stem.lower() if value else ""


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def safe_read_csv(path: Path, **kwargs) -> pd.DataFrame:
    return pd.read_csv(path, low_memory=False, **kwargs)


def write_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def json_safe(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (dt.datetime, dt.date)):
        return obj.isoformat()
    return obj


def write_json(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=json_safe)


def save_fig(fig, filename: str, title: str, records_analysed: str, method: str, limitation: str, figure_register: list[dict]) -> Path:
    path = FIGURE_DIR / filename
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    figure_register.append({
        "figure_file": filename,
        "title": title,
        "records_analysed": records_analysed,
        "method": method,
        "scientific_interpretation": title,
        "limitations": limitation,
        "relative_path": str(path.relative_to(PROJECT_ROOT)),
    })
    return path


def display_frame(title: str, df: pd.DataFrame, n: int = 10) -> None:
    display(Markdown(f"### {title}"))
    display(df.head(n))

## Input Verification

The notebook now checks whether the mandatory source files are present. The large image archive is not uploaded into this chat; it should remain in the local Jupyter project folder. This is safer for a large archive because the notebook can stream images from the ZIP file one at a time.

Mandatory Phase 6 inputs:

- `images.zip` - original image archive from Zenodo;
- `image_index.csv` - source metadata table for the images.

Supporting checkpoint inputs:

- Phase 4 linkage/evidence files;
- Phase 5 validation, manifest, and boundary files.

The supporting files help preserve scientific boundaries, but the raw image evidence still comes from the image archive and the source image index.

In [9]:
# Resolve input files and checkpoint folders.

IMAGE_ZIP_PATH = RAW_DIR / CONFIG["archive_name"]

image_index_candidates = [RAW_DIR / CONFIG["image_index_name"]]
image_index_candidates.extend(sorted(RAW_DIR.glob("image_index*.csv")))
image_index_candidates = _unique_existing_paths(image_index_candidates)
IMAGE_INDEX_PATH = image_index_candidates[0] if image_index_candidates else RAW_DIR / CONFIG["image_index_name"]

PHASE4_DIR = PROCESSED_DIR / "record_linkage_and_analysis_ready_evidence_layer"
PHASE5_DIR = PROCESSED_DIR / "hydrogel_rheology_and_printing_context_analysis"

phase4_files = {
    "phase4_validation_summary": PHASE4_DIR / "phase4_validation_summary.csv",
    "phase4_run_manifest": PHASE4_DIR / "phase4_run_manifest.json",
    "qc_linkage_summary": PHASE4_DIR / "qc_linkage_summary.csv",
    "candidate_rheology_image_linkage": PHASE4_DIR / "candidate_rheology_image_linkage.csv",
    "image_metadata_linkage_layer": PHASE4_DIR / "image_metadata_linkage_layer.csv",
    "unresolved_linkage_review_items": PHASE4_DIR / "unresolved_linkage_review_items.csv",
}

phase5_files = {
    "phase5_validation_summary": PHASE5_DIR / "phase5_validation_summary.csv",
    "phase5_run_manifest": PHASE5_DIR / "phase5_run_manifest.json",
    "phase5_input_file_resolution": PHASE5_DIR / "phase5_input_file_resolution.csv",
    "phase5_qc_and_boundary_summary": PHASE5_DIR / "phase5_qc_and_boundary_summary.csv",
    "phase5_analysis_boundary_table": PHASE5_DIR / "phase5_analysis_boundary_table.csv",
    "phase5_data_dictionary": PHASE5_DIR / "phase5_data_dictionary.csv",
}

input_records = []

def add_input_record(logical_name, path, mandatory, role):
    path = Path(path)
    exists = path.exists()
    input_records.append({
        "logical_name": logical_name,
        "expected_path": str(path),
        "relative_path": str(path.relative_to(PROJECT_ROOT)) if exists and path.is_relative_to(PROJECT_ROOT) else str(path),
        "mandatory": bool(mandatory),
        "role": role,
        "exists": bool(exists),
        "file_size_bytes": int(path.stat().st_size) if exists and path.is_file() else None,
        "sha256": sha256_file(path) if exists and path.is_file() and path.stat().st_size < 50_000_000 else None,
        "status": "available" if exists else ("missing_required" if mandatory else "missing_optional"),
    })

add_input_record("images_zip", IMAGE_ZIP_PATH, True, "Original raw image archive")
add_input_record("image_index_csv", IMAGE_INDEX_PATH, True, "Source image metadata table")

for name, path in phase4_files.items():
    add_input_record(name, path, False, "Phase 4 supporting evidence and linkage boundary file")
for name, path in phase5_files.items():
    add_input_record(name, path, False, "Phase 5 supporting validation and analysis-boundary file")

input_verification = pd.DataFrame(input_records)
write_csv(input_verification, OUTPUT_DIR / "phase6_input_verification.csv")

missing_required = input_verification.query("mandatory == True and exists == False")
display_frame("Input Verification", input_verification, n=30)

if not missing_required.empty:
    missing_list = "\n".join(f"- {row.logical_name}: {row.expected_path}" for row in missing_required.itertuples())
    raise FileNotFoundError(
        "Phase 6 cannot run until the mandatory input files are present.\n\n"
        "Missing required files:\n"
        f"{missing_list}\n\n"
        "Place images.zip at data/raw/zenodo_19602891/images.zip and image_index.csv in the same raw folder, "
        "then restart the kernel and run all cells again."
    )

### Input Verification

,logical_name,expected_path,relative_path,mandatory,role,exists,file_size_bytes,sha256,status
0,images_zip,C:\Users\sufer\Documents\hydrogel-bioink\data\raw\zenodo_19602891\images.zip,data\raw\zenodo_19602891\images.zip,True,Original raw image archive,True,482098042,None,available
1,image_index_csv,C:\Users\sufer\Documents\hydrogel-bioink\data\raw\zenodo_19602891\image_index.csv,data\raw\zenodo_19602891\image_index.csv,True,Source image metadata table,True,37937,6e51d222d5434e271e4757ea750b8b4b002df13f056c21a5037d0e0b33a7b2c5,available
2,phase4_validation_summary,C:\Users\sufer\Documents\hydrogel-bioink\data\processed\record_linkage_and_analysis_ready_evidence_layer\phase4_validation_summary.csv,data\processed\record_linkage_and_analysis_ready_evidence_layer\phase4_validation_summary.csv,False,Phase 4 supporting evidence and linkage boundary file,True,3789,34e65e2e7449b9ec02cdc42cefca5dc65d8b797f74331d8567cd20232e559d38,available
3,phase4_run_manifest,C:\Users\sufer\Documents\hydrogel-bioink\data\processed\record_linkage_and_analysis_ready_evidence_layer\phase4_run_manifest.json,data\processed\record_linkage_and_analysis_ready_evidence_layer\phase4_run_manifest.json,False,Phase 4 supporting evidence and linkage boundary file,True,21385,c7e5760997a72afc9f338b89f73aa496d6897ec85c383680a615e8747644b0a9,available
4,qc_linkage_summary,C:\Users\sufer\Documents\hydrogel-bioink\data\processed\record_linkage_and_analysis_ready_evidence_layer\qc_linkage_summary.csv,data\processed\record_linkage_and_analysis_ready_evidence_layer\qc_linkage_summary.csv,False,Phase 4 supporting evidence and linkage boundary file,True,8779,116548169a6f849aef82b48753f493ce2ca03a5d171b7f12e1d0f9106b4a6671,available
5,candidate_rheology_image_linkage,C:\Users\sufer\Documents\hydrogel-bioink\data\processed\record_linkage_and_analysis_ready_evidence_layer\candidate_rheology_image_linkage.csv,data\processed\record_linkage_and_analysis_ready_evidence_layer\candidate_rheology_image_linkage.csv,False,Phase 4 supporting evidence and linkage boundary file,True,22937,f6314d467ccf0c516d08475d287b1a9e94b430f025ea1f584227049d41e58e42,available
6,image_metadata_linkage_layer,C:\Users\sufer\Documents\hydrogel-bioink\data\processed\record_linkage_and_analysis_ready_evidence_layer\image_metadata_linkage_layer.csv,data\processed\record_linkage_and_analysis_ready_evidence_layer\image_metadata_linkage_layer.csv,False,Phase 4 supporting evidence and linkage boundary file,True,549169,d9155262e0126c0dbaf0b916b53148298969a9ba729c2ab9d30b547604beea5d,available
7,unresolved_linkage_review_items,C:\Users\sufer\Documents\hydrogel-bioink\data\processed\record_linkage_and_analysis_ready_evidence_layer\unresolved_linkage_review_items.csv,data\processed\record_linkage_and_analysis_ready_evidence_layer\unresolved_linkage_review_items.csv,False,Phase 4 supporting evidence and linkage boundary file,True,2810,a6ac675bf398343b459e19bc99563e618bd2ccee3d172dd87161261b5d779ec4,available
8,phase5_validation_summary,C:\Users\sufer\Documents\hydrogel-bioink\data\processed\hydrogel_rheology_and_printing_context_analysis\phase5_validation_summary.csv,data\processed\hydrogel_rheology_and_printing_context_analysis\phase5_validation_summary.csv,False,Phase 5 supporting validation and analysis-boundary file,True,16189,5b14d5bcb03aef8b986d15c58ab3e8131ac9771a7a636549d3777c67b469d60b,available
9,phase5_run_manifest,C:\Users\sufer\Documents\hydrogel-bioink\data\processed\hydrogel_rheology_and_printing_context_analysis\phase5_run_manifest.json,data\processed\hydrogel_rheology_and_printing_context_analysis\phase5_run_manifest.json,False,Phase 5 supporting validation and analysis-boundary file,True,19519,ff78705cacb78ee43580ed7f7e44f547fe14b79b5b6c2b4b40a994787bc9fac1,available


In [10]:
# Load the source image index and optional supporting checkpoint tables.

image_index = safe_read_csv(IMAGE_INDEX_PATH)
image_index.columns = [str(c).strip() for c in image_index.columns]
image_index["source_csv_record"] = np.arange(1, len(image_index) + 1)

supporting_tables = {}
for name, path in {**phase4_files, **phase5_files}.items():
    if path.exists() and path.suffix.lower() == ".csv":
        supporting_tables[name] = safe_read_csv(path)
    elif path.exists() and path.suffix.lower() == ".json":
        with open(path, "r", encoding="utf-8") as f:
            supporting_tables[name] = json.load(f)

print("Image index path:", IMAGE_INDEX_PATH)
print("Image index records:", len(image_index))
print("Image index columns:", list(image_index.columns))
print("Loaded supporting checkpoint objects:", sorted(supporting_tables))

display_frame("Image Index Preview", image_index, n=5)

Image index path: C:\Users\sufer\Documents\hydrogel-bioink\data\raw\zenodo_19602891\image_index.csv
Image index records: 1031
Image index columns: ['image_id', 'date', 'ink_type_ph', 'shape', 'intensity', 'completeness_sam', 'contain_sps', 'source_csv_record']
Loaded supporting checkpoint objects: ['candidate_rheology_image_linkage', 'image_metadata_linkage_layer', 'phase4_run_manifest', 'phase4_validation_summary', 'phase5_analysis_boundary_table', 'phase5_data_dictionary', 'phase5_input_file_resolution', 'phase5_qc_and_boundary_summary', 'phase5_run_manifest', 'phase5_validation_summary', 'qc_linkage_summary', 'unresolved_linkage_review_items']


### Image Index Preview

,image_id,date,ink_type_ph,shape,intensity,completeness_sam,contain_sps,source_csv_record
0,7586,2025/05/28,ALG2.0,grid,1,NaN,NaN,1
1,7587,2025/05/28,ALG2.0,grid,1,NaN,NaN,2
2,7588,2025/05/28,ALG2.0,grid,1,NaN,NaN,3
3,7589,2025/05/28,ALG2.0,grid,50,NaN,NaN,4
4,7590,2025/05/28,ALG2.0,grid,50,no,NaN,5


## Image Metadata as Scientific Evidence

`image_index.csv` is not just a convenient spreadsheet. It is a metadata table that tells us how the images are described by the source dataset. In this project, fields such as `ink_type_ph`, `shape`, `intensity`, `completeness_sam`, and `contain_sps` may later help organise image observations by formulation, printed geometry, or acquisition context.

However, metadata fields are not automatically experimental truth. A label may be missing, ambiguous, repeated, or not sufficient to establish physical-sample identity. Phase 6 therefore profiles completeness and uniqueness before linking any image to any downstream interpretation.

In [12]:
# Profile source image metadata without altering original values.

metadata_columns = [c for c in image_index.columns if c != "source_csv_record"]

image_index_profile_records = []
for column in metadata_columns:
    series = image_index[column]
    image_index_profile_records.append({
        "field": column,
        "records": int(len(series)),
        "present": int(series.notna().sum()),
        "missing": int(series.isna().sum()),
        "missing_fraction": float(series.isna().mean()),
        "unique_non_missing_values": int(series.dropna().astype(str).nunique()),
        "example_values": "; ".join(series.dropna().astype(str).drop_duplicates().head(8).tolist()),
    })

image_index_profile = pd.DataFrame(image_index_profile_records)
write_csv(image_index_profile, OUTPUT_DIR / "phase6_image_index_profile.csv")

if "image_id" in image_index.columns:
    duplicate_image_ids = image_index[image_index["image_id"].astype(str).duplicated(keep=False)].copy()
else:
    duplicate_image_ids = pd.DataFrame()

write_csv(duplicate_image_ids, OUTPUT_DIR / "phase6_duplicate_image_ids.csv")

display_frame("Image Metadata Field Profile", image_index_profile, n=30)
print("Duplicate image_id records:", len(duplicate_image_ids))

### Image Metadata Field Profile

,field,records,present,missing,missing_fraction,unique_non_missing_values,example_values
0,image_id,1031,1031,0,0.000000,1031,7586; 7587; 7588; 7589; 7590; 7591; 7592; 7593
1,date,1031,1031,0,0.000000,18,2025/05/28; 2025/05/29; 2025/06/05; 2025/06/10; 2025/06/12; 2025/06/26; 2025/07/01; 2025/07/02
2,ink_type_ph,1031,1031,0,0.000000,48,ALG2.0; ALG1.0; ALG3.0; ALG4.0; ALG5.0; ALG1.5; ALG2.5; HA1.0
3,shape,1031,1031,0,0.000000,4,grid; star; ichou; nose
4,intensity,1031,1031,0,0.000000,7,1; 50; 100; 0; 75; 25; 10
5,completeness_sam,1031,88,943,0.914646,1,no
6,contain_sps,1031,109,922,0.894277,1,no


Duplicate image_id records: 0


## ZIP Archive Inventory

A ZIP archive is a container. It may include image files, folders, metadata, hidden operating-system files, or unexpected extras. An archive inventory gives us a source-level map before we open or analyse any image.

For a materials-data project, this matters because a file path is evidence. If a printed-grid image cannot be traced back to the original archive member, then later feature extraction is not properly auditable.

In [14]:
# Inventory the ZIP archive without extracting it to disk.

if not zipfile.is_zipfile(IMAGE_ZIP_PATH):
    raise ValueError(f"The file exists but is not a valid ZIP archive: {IMAGE_ZIP_PATH}")

zip_integrity_bad_member = None
zip_records = []
zip_test_started = time.time()

with zipfile.ZipFile(IMAGE_ZIP_PATH, "r") as zf:
    if CONFIG["run_zip_integrity_test"]:
        zip_integrity_bad_member = zf.testzip()
    for info in zf.infolist():
        archive_path = info.filename.replace("\\", "/")
        suffix = Path(archive_path).suffix.lower()
        zip_records.append({
            "archive_path": archive_path,
            "file_name": Path(archive_path).name,
            "directory": str(Path(archive_path).parent).replace(".", ""),
            "suffix": suffix,
            "is_directory": bool(info.is_dir()),
            "is_image_candidate": bool((not info.is_dir()) and suffix in IMAGE_SUFFIXES),
            "file_size_bytes": int(info.file_size),
            "compressed_size_bytes": int(info.compress_size),
            "crc": f"{info.CRC:08x}",
            "zip_modified_datetime": "%04d-%02d-%02d %02d:%02d:%02d" % info.date_time,
            "archive_path_norm": normalise_text(archive_path),
            "file_name_norm": normalise_filename(archive_path),
            "stem_norm": normalise_stem(archive_path),
        })

zip_inventory = pd.DataFrame(zip_records)
image_file_inventory = zip_inventory.query("is_image_candidate == True").copy()

write_csv(zip_inventory, OUTPUT_DIR / "phase6_zip_file_inventory.csv")
write_csv(image_file_inventory, OUTPUT_DIR / "phase6_image_file_inventory.csv")

print("ZIP integrity bad member:", zip_integrity_bad_member)
print("ZIP integrity check seconds:", round(time.time() - zip_test_started, 2))
print("Archive members:", len(zip_inventory))
print("Image candidate files:", len(image_file_inventory))
print("Image suffix counts:")
display(image_file_inventory["suffix"].value_counts(dropna=False).rename_axis("suffix").reset_index(name="count"))

display_frame("Image File Inventory Preview", image_file_inventory, n=10)

ZIP integrity bad member: None
ZIP integrity check seconds: 2.08
Archive members: 1032
Image candidate files: 1031
Image suffix counts:


,suffix,count
0,.png,1031


### Image File Inventory Preview

,archive_path,file_name,directory,suffix,is_directory,is_image_candidate,file_size_bytes,compressed_size_bytes,crc,zip_modified_datetime,archive_path_norm,file_name_norm,stem_norm
1,images/resized_IMG_0304.png,resized_IMG_0304.png,images,.png,False,True,500482,500637,48a8e016,2025-11-11 20:46:14,images/resized_img_0304.png,resized_img_0304.png,resized_img_0304
2,images/resized_IMG_0305.png,resized_IMG_0305.png,images,.png,False,True,494486,494641,6ff527b3,2025-11-11 20:46:14,images/resized_img_0305.png,resized_img_0305.png,resized_img_0305
3,images/resized_IMG_0306.png,resized_IMG_0306.png,images,.png,False,True,495449,495604,9a812a5f,2025-11-11 20:46:14,images/resized_img_0306.png,resized_img_0306.png,resized_img_0306
4,images/resized_IMG_0307.png,resized_IMG_0307.png,images,.png,False,True,485163,485313,9372f359,2025-11-11 20:46:14,images/resized_img_0307.png,resized_img_0307.png,resized_img_0307
5,images/resized_IMG_0308.png,resized_IMG_0308.png,images,.png,False,True,471346,471491,f10a2063,2025-11-11 20:46:12,images/resized_img_0308.png,resized_img_0308.png,resized_img_0308
6,images/resized_IMG_0309.png,resized_IMG_0309.png,images,.png,False,True,519283,519397,d8a2a9af,2025-11-11 20:46:12,images/resized_img_0309.png,resized_img_0309.png,resized_img_0309
7,images/resized_IMG_0311.png,resized_IMG_0311.png,images,.png,False,True,499080,499235,33984b19,2025-11-11 20:46:14,images/resized_img_0311.png,resized_img_0311.png,resized_img_0311
8,images/resized_IMG_0312.png,resized_IMG_0312.png,images,.png,False,True,507029,507184,b64e856f,2025-11-11 20:46:14,images/resized_img_0312.png,resized_img_0312.png,resized_img_0312
9,images/resized_IMG_0314.png,resized_IMG_0314.png,images,.png,False,True,508074,508229,c1affabf,2025-11-11 20:46:14,images/resized_img_0314.png,resized_img_0314.png,resized_img_0314
10,images/resized_IMG_0315.png,resized_IMG_0315.png,images,.png,False,True,501415,501570,a509708c,2025-11-11 20:46:14,images/resized_img_0315.png,resized_img_0315.png,resized_img_0315


## Linking Image Metadata to Actual Files

Linkage is the step where source metadata records are compared with archive files. This is not the same as proving physical-sample identity. A successful file-name match means only that an image-index record can be associated with an archived file under a documented matching rule.

This distinction protects the science. File-name similarity can support an image evidence layer, but it does not prove that a rheology measurement and a printed-image observation came from the same physical hydrogel sample.

In [16]:
# Link image_index records to archive image files using transparent matching rules.

# Optional Phase 4 expected filenames, if available.
phase4_expected = pd.DataFrame()
if "image_metadata_linkage_layer" in supporting_tables:
    temp = supporting_tables["image_metadata_linkage_layer"].copy()
    if {"image_id", "expected_filename"}.issubset(temp.columns):
        phase4_expected = temp[["image_id", "expected_filename"]].drop_duplicates()
        phase4_expected["image_id_norm"] = phase4_expected["image_id"].map(normalise_text)
        phase4_expected = phase4_expected.drop_duplicates("image_id_norm")

working_index = image_index.copy()
if "image_id" not in working_index.columns:
    raise KeyError("image_index.csv must contain an image_id column for Phase 6 linkage.")

working_index["image_id_norm"] = working_index["image_id"].map(normalise_text)
working_index["image_id_stem_norm"] = working_index["image_id"].map(normalise_stem)

if not phase4_expected.empty:
    working_index = working_index.merge(
        phase4_expected[["image_id_norm", "expected_filename"]],
        on="image_id_norm",
        how="left",
    )
else:
    working_index["expected_filename"] = ""

file_name_map = defaultdict(list)
stem_map = defaultdict(list)
archive_path_map = defaultdict(list)
for row in image_file_inventory.itertuples(index=False):
    file_name_map[row.file_name_norm].append(row.archive_path)
    stem_map[row.stem_norm].append(row.archive_path)
    archive_path_map[row.archive_path_norm].append(row.archive_path)

archive_paths_norm_list = image_file_inventory["archive_path_norm"].tolist()
archive_file_names_norm_list = image_file_inventory["file_name_norm"].tolist()

linkage_records = []
for row in working_index.itertuples(index=False):
    image_id = getattr(row, "image_id")
    image_id_norm = getattr(row, "image_id_norm")
    image_id_stem_norm = getattr(row, "image_id_stem_norm")
    expected_filename = getattr(row, "expected_filename", "")
    expected_name_norm = normalise_filename(expected_filename)
    expected_stem_norm = normalise_stem(expected_filename)

    matched_paths = []
    match_method = "no_archive_match"

    if expected_name_norm and expected_name_norm in file_name_map:
        matched_paths = file_name_map[expected_name_norm]
        match_method = "phase4_expected_filename_exact_file_name"
    elif expected_stem_norm and expected_stem_norm in stem_map:
        matched_paths = stem_map[expected_stem_norm]
        match_method = "phase4_expected_filename_exact_stem"
    elif image_id_norm and image_id_norm in file_name_map:
        matched_paths = file_name_map[image_id_norm]
        match_method = "image_id_exact_file_name"
    elif image_id_stem_norm and image_id_stem_norm in stem_map:
        matched_paths = stem_map[image_id_stem_norm]
        match_method = "image_id_exact_stem"
    elif image_id_norm:
        contains = [p for p, p_norm in zip(image_file_inventory["archive_path"], archive_paths_norm_list) if image_id_norm in p_norm]
        if contains:
            matched_paths = contains
            match_method = "image_id_token_in_archive_path"

    linkage_records.append({
        "source_csv_record": getattr(row, "source_csv_record"),
        "image_id": image_id,
        "expected_filename": expected_filename,
        "matched_archive_paths": " | ".join(matched_paths),
        "matched_archive_path_count": int(len(matched_paths)),
        "primary_archive_path": matched_paths[0] if len(matched_paths) == 1 else "",
        "linkage_status": "matched_one_file" if len(matched_paths) == 1 else ("matched_multiple_files_review" if len(matched_paths) > 1 else "not_matched_review"),
        "match_method": match_method,
        "scientific_boundary_note": "File linkage supports image-file traceability only; it does not prove physical-sample identity or printability outcome.",
    })

image_index_to_archive_linkage = pd.DataFrame(linkage_records)
write_csv(image_index_to_archive_linkage, OUTPUT_DIR / "phase6_image_index_to_archive_linkage.csv")

matched_archive_paths = set(
    p for paths in image_index_to_archive_linkage["matched_archive_paths"].dropna().astype(str)
    for p in paths.split(" | ") if p
)
archive_only_images = image_file_inventory[~image_file_inventory["archive_path"].isin(matched_archive_paths)].copy()
write_csv(archive_only_images, OUTPUT_DIR / "phase6_archive_only_images.csv")

print("Index records:", len(image_index_to_archive_linkage))
print("Matched one file:", int((image_index_to_archive_linkage["linkage_status"] == "matched_one_file").sum()))
print("Matched multiple files:", int((image_index_to_archive_linkage["linkage_status"] == "matched_multiple_files_review").sum()))
print("Not matched:", int((image_index_to_archive_linkage["linkage_status"] == "not_matched_review").sum()))
print("Archive-only image files:", len(archive_only_images))

display_frame("Image Index to Archive Linkage Preview", image_index_to_archive_linkage, n=10)

Index records: 1031
Matched one file: 1031
Matched multiple files: 0
Not matched: 0
Archive-only image files: 0


### Image Index to Archive Linkage Preview

,source_csv_record,image_id,expected_filename,matched_archive_paths,matched_archive_path_count,primary_archive_path,linkage_status,match_method,scientific_boundary_note
0,1,7586,resized_IMG_7586.png,images/resized_IMG_7586.png,1,images/resized_IMG_7586.png,matched_one_file,phase4_expected_filename_exact_file_name,File linkage supports image-file traceability only; it does not prove physical-sample identity or printability outcome.
1,2,7587,resized_IMG_7587.png,images/resized_IMG_7587.png,1,images/resized_IMG_7587.png,matched_one_file,phase4_expected_filename_exact_file_name,File linkage supports image-file traceability only; it does not prove physical-sample identity or printability outcome.
2,3,7588,resized_IMG_7588.png,images/resized_IMG_7588.png,1,images/resized_IMG_7588.png,matched_one_file,phase4_expected_filename_exact_file_name,File linkage supports image-file traceability only; it does not prove physical-sample identity or printability outcome.
3,4,7589,resized_IMG_7589.png,images/resized_IMG_7589.png,1,images/resized_IMG_7589.png,matched_one_file,phase4_expected_filename_exact_file_name,File linkage supports image-file traceability only; it does not prove physical-sample identity or printability outcome.
4,5,7590,resized_IMG_7590.png,images/resized_IMG_7590.png,1,images/resized_IMG_7590.png,matched_one_file,phase4_expected_filename_exact_file_name,File linkage supports image-file traceability only; it does not prove physical-sample identity or printability outcome.
5,6,7591,resized_IMG_7591.png,images/resized_IMG_7591.png,1,images/resized_IMG_7591.png,matched_one_file,phase4_expected_filename_exact_file_name,File linkage supports image-file traceability only; it does not prove physical-sample identity or printability outcome.
6,7,7592,resized_IMG_7592.png,images/resized_IMG_7592.png,1,images/resized_IMG_7592.png,matched_one_file,phase4_expected_filename_exact_file_name,File linkage supports image-file traceability only; it does not prove physical-sample identity or printability outcome.
7,8,7593,resized_IMG_7593.png,images/resized_IMG_7593.png,1,images/resized_IMG_7593.png,matched_one_file,phase4_expected_filename_exact_file_name,File linkage supports image-file traceability only; it does not prove physical-sample identity or printability outcome.
8,9,7594,resized_IMG_7594.png,images/resized_IMG_7594.png,1,images/resized_IMG_7594.png,matched_one_file,phase4_expected_filename_exact_file_name,File linkage supports image-file traceability only; it does not prove physical-sample identity or printability outcome.
9,10,7595,resized_IMG_7595.png,images/resized_IMG_7595.png,1,images/resized_IMG_7595.png,matched_one_file,phase4_expected_filename_exact_file_name,File linkage supports image-file traceability only; it does not prove physical-sample identity or printability outcome.


## Image Arrays, Pixels, Channels and Hydrogel Images

When Python opens an image, it usually becomes an array. A grayscale image may have one intensity value per pixel. An RGB image has three channels per pixel: red, green and blue. The notebook converts source images into small thumbnails for feature extraction so that a large archive can be processed without loading the whole ZIP into memory.

In hydrogel printing, simple image descriptors help us ask careful evidence questions:

- **Dimensions:** are the image sizes consistent, or were some files exported differently?
- **Brightness and contrast:** are some images unusually dark, bright, faint or blank-like?
- **Foreground area:** how much of the thumbnail differs from the estimated background?
- **Edge density:** are there many intensity changes, which may reflect lines, grids, texture, shadows or noise?
- **Duplicate cues:** do exact file hashes or thumbnail hashes suggest repeated visual evidence?

These are data-quality and exploratory questions. They do not prove hydrogel printability. In polymer terms, printability depends on formulation and process behaviour such as viscosity, crosslinking kinetics, gel strength, filament shape retention and post-print stability. A photograph can give useful visible evidence, but it is not the same thing as rheology, microscopy, viability testing or mechanical testing.


In [18]:
# Image feature extraction helpers. Images are processed one at a time.


def pil_to_rgb_thumbnail(img: Image.Image, max_dim: int) -> Image.Image:
    rgb = img.convert("RGB")
    rgb.thumbnail((max_dim, max_dim), Image.Resampling.LANCZOS)
    return rgb


def rgb_array_to_gray(arr_rgb: np.ndarray) -> np.ndarray:
    arr = arr_rgb.astype(np.float32) / 255.0
    return 0.2126 * arr[:, :, 0] + 0.7152 * arr[:, :, 1] + 0.0722 * arr[:, :, 2]


def border_values(gray: np.ndarray) -> np.ndarray:
    if gray.size == 0:
        return np.array([], dtype=np.float32)
    top = gray[0, :]
    bottom = gray[-1, :]
    left = gray[:, 0]
    right = gray[:, -1]
    return np.concatenate([top, bottom, left, right])


def foreground_mask_from_border(gray: np.ndarray) -> tuple[np.ndarray, float, float, str]:
    """Create a cautious foreground mask using border intensity as a background estimate."""
    borders = border_values(gray)
    if borders.size == 0:
        return np.zeros_like(gray, dtype=bool), np.nan, np.nan, "empty_image"

    background = float(np.nanmedian(borders))
    mad = float(np.nanmedian(np.abs(borders - background)))
    threshold = max(0.08, 3.0 * mad)
    diff = np.abs(gray - background)
    mask = diff > threshold
    return mask, background, threshold, "border_difference_threshold"


def otsu_threshold_numpy(gray: np.ndarray, bins: int = 256) -> float:
    """Compute an Otsu-style grayscale threshold using only NumPy."""
    values = np.asarray(gray, dtype=np.float32).ravel()
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan

    v_min = float(values.min())
    v_max = float(values.max())
    if not v_max > v_min:
        return v_min

    hist, bin_edges = np.histogram(values, bins=bins, range=(v_min, v_max))
    hist = hist.astype(np.float64)
    total = hist.sum()
    if total <= 0:
        return np.nan

    centres = (bin_edges[:-1] + bin_edges[1:]) / 2.0
    prob = hist / total
    omega = np.cumsum(prob)
    mu = np.cumsum(prob * centres)
    mu_total = mu[-1]
    denom = omega * (1.0 - omega)

    between_class_variance = np.full_like(centres, np.nan, dtype=np.float64)
    valid = denom > 0
    between_class_variance[valid] = ((mu_total * omega[valid] - mu[valid]) ** 2) / denom[valid]

    if np.all(np.isnan(between_class_variance)):
        return float(np.nanmedian(values))
    return float(centres[int(np.nanargmax(between_class_variance))])


def otsu_threshold_safe(gray: np.ndarray) -> tuple[float, str]:
    if gray.size == 0:
        return np.nan, "empty_image"
    if SKIMAGE_AVAILABLE and threshold_otsu is not None:
        try:
            return float(threshold_otsu(gray)), "scikit_image_threshold_otsu"
        except Exception:
            pass
    return otsu_threshold_numpy(gray), "numpy_histogram_otsu"


def edge_density_simple(gray: np.ndarray) -> float:
    if min(gray.shape) < 3:
        return np.nan
    gy, gx = np.gradient(gray.astype(np.float32))
    grad = np.sqrt(gx * gx + gy * gy)
    threshold = float(np.nanmean(grad) + np.nanstd(grad))
    if not np.isfinite(threshold) or threshold <= 0:
        return 0.0
    return float((grad > threshold).mean())


def dhash_hex(img: Image.Image) -> str:
    small = img.convert("L").resize((9, 8), Image.Resampling.LANCZOS)
    arr = np.asarray(small, dtype=np.int16)
    diff = arr[:, 1:] > arr[:, :-1]
    bits = "".join("1" if v else "0" for v in diff.flatten())
    return f"{int(bits, 2):016x}"


def image_channels(mode: str) -> int | None:
    mapping = {"1": 1, "L": 1, "P": 1, "LA": 2, "RGB": 3, "RGBA": 4, "CMYK": 4, "I": 1, "F": 1}
    return mapping.get(mode)


def extract_image_record(zf: zipfile.ZipFile, archive_path: str) -> dict:
    record = {"archive_path": archive_path, "readable": False}
    try:
        with zf.open(archive_path, "r") as f:
            data = f.read()
        if CONFIG["compute_image_sha256"]:
            record["sha256"] = sha256_bytes(data)
        else:
            record["sha256"] = None

        with Image.open(BytesIO(data)) as img:
            record.update({
                "readable": True,
                "image_format": img.format,
                "width_px": int(img.width),
                "height_px": int(img.height),
                "pixel_count": int(img.width * img.height),
                "mode": img.mode,
                "channel_count": image_channels(img.mode),
            })

            rgb_thumb = pil_to_rgb_thumbnail(img, CONFIG["feature_dimension_px"])
            rgb_arr = np.asarray(rgb_thumb)
            gray = rgb_array_to_gray(rgb_arr)
            mask, bg, fg_threshold, fg_method = foreground_mask_from_border(gray)
            otsu_threshold, otsu_method = otsu_threshold_safe(gray)

            record.update({
                "feature_image_width_px": int(rgb_thumb.width),
                "feature_image_height_px": int(rgb_thumb.height),
                "gray_mean": float(np.nanmean(gray)),
                "gray_std": float(np.nanstd(gray)),
                "gray_min": float(np.nanmin(gray)),
                "gray_p05": float(np.nanpercentile(gray, 5)),
                "gray_median": float(np.nanmedian(gray)),
                "gray_p95": float(np.nanpercentile(gray, 95)),
                "gray_max": float(np.nanmax(gray)),
                "red_mean": float(np.nanmean(rgb_arr[:, :, 0] / 255.0)),
                "green_mean": float(np.nanmean(rgb_arr[:, :, 1] / 255.0)),
                "blue_mean": float(np.nanmean(rgb_arr[:, :, 2] / 255.0)),
                "background_gray_estimate": bg,
                "foreground_threshold": fg_threshold,
                "foreground_method": fg_method,
                "foreground_area_fraction": float(mask.mean()) if mask.size else np.nan,
                "otsu_threshold": otsu_threshold,
                "otsu_method": otsu_method,
                "edge_density_simple": edge_density_simple(gray),
                "thumbnail_dhash": dhash_hex(rgb_thumb),
                "blank_like_flag": bool(float(np.nanstd(gray)) < 0.01),
                "low_contrast_flag": bool(float(np.nanstd(gray)) < 0.05),
                "very_dark_flag": bool(float(np.nanmean(gray)) < 0.10),
                "very_bright_flag": bool(float(np.nanmean(gray)) > 0.90),
                "feature_interpretation_boundary": "Computed image descriptors support exploratory audit only; they are not validated printability or morphology labels.",
            })
    except Exception as exc:
        record.update({
            "readable": False,
            "error_type": type(exc).__name__,
            "error_message": str(exc)[:500],
        })
    return record


In [19]:
# Read image files one at a time from the ZIP archive and extract basic properties/features.

feature_records = []
start = time.time()
archive_paths = image_file_inventory["archive_path"].tolist()

with zipfile.ZipFile(IMAGE_ZIP_PATH, "r") as zf:
    for i, archive_path in enumerate(archive_paths, start=1):
        feature_records.append(extract_image_record(zf, archive_path))
        if i % CONFIG["progress_every_n_images"] == 0 or i == len(archive_paths):
            elapsed = time.time() - start
            print(f"Processed {i}/{len(archive_paths)} image files in {elapsed:.1f} seconds")

image_integrity_properties = pd.DataFrame(feature_records)

# Merge file-size and suffix metadata back into the image-property table.
image_integrity_properties = image_integrity_properties.merge(
    image_file_inventory[["archive_path", "file_name", "suffix", "file_size_bytes", "crc", "zip_modified_datetime"]],
    on="archive_path",
    how="left",
)

write_csv(image_integrity_properties, OUTPUT_DIR / "phase6_image_integrity_properties.csv")

feature_cols = [
    "archive_path", "sha256", "thumbnail_dhash", "gray_mean", "gray_std", "gray_min", "gray_p05", "gray_median",
    "gray_p95", "gray_max", "red_mean", "green_mean", "blue_mean", "foreground_area_fraction",
    "edge_density_simple", "blank_like_flag", "low_contrast_flag", "very_dark_flag", "very_bright_flag",
    "feature_interpretation_boundary",
]
phase6_image_feature_summary = image_integrity_properties[[c for c in feature_cols if c in image_integrity_properties.columns]].copy()
write_csv(phase6_image_feature_summary, OUTPUT_DIR / "phase6_image_feature_summary.csv")

print("Readable images:", int(image_integrity_properties["readable"].sum()))
print("Unreadable images:", int((~image_integrity_properties["readable"]).sum()))
display_frame("Image Integrity and Feature Preview", image_integrity_properties, n=10)

Processed 100/1031 image files in 14.6 seconds
Processed 200/1031 image files in 28.4 seconds
Processed 300/1031 image files in 42.8 seconds
Processed 400/1031 image files in 55.7 seconds
Processed 500/1031 image files in 69.1 seconds
Processed 600/1031 image files in 81.8 seconds
Processed 700/1031 image files in 95.7 seconds
Processed 800/1031 image files in 109.8 seconds
Processed 900/1031 image files in 123.8 seconds
Processed 1000/1031 image files in 137.7 seconds
Processed 1031/1031 image files in 141.7 seconds
Readable images: 1031
Unreadable images: 0


### Image Integrity and Feature Preview

,archive_path,readable,sha256,image_format,width_px,height_px,pixel_count,mode,channel_count,feature_image_width_px,feature_image_height_px,gray_mean,gray_std,gray_min,gray_p05,gray_median,gray_p95,gray_max,red_mean,green_mean,blue_mean,background_gray_estimate,foreground_threshold,foreground_method,foreground_area_fraction,otsu_threshold,otsu_method,edge_density_simple,thumbnail_dhash,blank_like_flag,low_contrast_flag,very_dark_flag,very_bright_flag,feature_interpretation_boundary,file_name,suffix,file_size_bytes,crc,zip_modified_datetime
0,images/resized_IMG_0304.png,True,26a9badf3cb49ddd8b042c6b7e41c002d000300afbfb411c3fbfc1ee822ca9b4,PNG,800,800,640000,RGB,3,512,512,0.634079,0.029236,0.347318,0.576761,0.643450,0.668951,0.703935,0.616110,0.652057,0.508900,0.643466,0.080000,border_difference_threshold,0.015285,0.622442,numpy_histogram_otsu,0.116306,6941450141095113,False,True,False,False,Computed image descriptors support exploratory audit only; they are not validated printability or morphology labels.,resized_IMG_0304.png,.png,500482,48a8e016,2025-11-11 20:46:14
1,images/resized_IMG_0305.png,True,b47e974c6e318d94170dfd3c49ae635afe5e7f93bfa1610fdb92fa0690ea3c25,PNG,800,800,640000,RGB,3,512,512,0.724466,0.030052,0.492082,0.659922,0.733060,0.757124,0.801010,0.728851,0.734398,0.613171,0.728284,0.080000,border_difference_threshold,0.024208,0.703867,numpy_histogram_otsu,0.105385,6949000540085010,False,True,False,False,Computed image descriptors support exploratory audit only; they are not validated printability or morphology labels.,resized_IMG_0305.png,.png,494486,6ff527b3,2025-11-11 20:46:14
2,images/resized_IMG_0306.png,True,0bb48e6a642115d40e1813f36f40cfdf5689c3a844c8e603c4ebb2701f68f6d5,PNG,800,800,640000,RGB,3,512,512,0.718483,0.027970,0.495794,0.655392,0.726868,0.747849,0.783652,0.721063,0.728074,0.615879,0.720724,0.080000,border_difference_threshold,0.017818,0.697632,numpy_histogram_otsu,0.105579,4949840548815032,False,True,False,False,Computed image descriptors support exploratory audit only; they are not validated printability or morphology labels.,resized_IMG_0306.png,.png,495449,9a812a5f,2025-11-11 20:46:14
3,images/resized_IMG_0307.png,True,68dcb6d80ccd7e72e9fdda07de5e56c76abd1ecdc886d3794dc1118628b17ead,PNG,800,800,640000,RGB,3,512,512,0.705865,0.030709,0.442994,0.638915,0.715670,0.737799,0.781744,0.708553,0.716338,0.594208,0.709813,0.080000,border_difference_threshold,0.026482,0.684485,numpy_histogram_otsu,0.106316,684941004a497111,False,True,False,False,Computed image descriptors support exploratory audit only; they are not validated printability or morphology labels.,resized_IMG_0307.png,.png,485163,9372f359,2025-11-11 20:46:14
4,images/resized_IMG_0308.png,True,79924505b2c988658b97abbce659c4036a0e0659055b31699f0248a892cc152f,PNG,800,800,640000,RGB,3,512,512,0.710833,0.029139,0.537127,0.646978,0.719875,0.741301,0.778016,0.717692,0.720529,0.594592,0.715953,0.080000,border_difference_threshold,0.025719,0.690035,numpy_histogram_otsu,0.105850,6d6d016d40493012,False,True,False,False,Computed image descriptors support exploratory audit only; they are not validated printability or morphology labels.,resized_IMG_0308.png,.png,471346,f10a2063,2025-11-11 20:46:12
5,images/resized_IMG_0309.png,True,130b06c4ff17edca901e20bb8b016d785df9aaf1ffab022e5391d47c5a3752da,PNG,800,800,640000,RGB,3,512,512,0.649191,0.082337,0.144776,0.475246,0.684045,0.713447,0.779433,0.680138,0.652311,0.527150,0.689068,0.080000,border_difference_threshold,0.200657,0.587301,numpy_histogram_otsu,0.050022,4c3396923315cc08,False,False,False,False,Computed image descriptors support exploratory audit only; they are not validated printability or morphology labels.,resized_IMG_0309.png,.png,519283,d8a2a9af,2025-11-11 20:46:12
6,images/resized_IMG_0311.png,True,12a58131ea1c5662fce99feaef0fba288fb5e95942fffa1d77523c2cf341b92b,PNG,800,800,640000,RGB,3,512,512,0.766607,0.239875,0.002501,0.340464,0.884848,0.966835,1.000000,0.813380,0.762795,0.666643,0.861617,0.196

## Duplicate Diagnostics and Review Flags

Duplicate detection has two meanings here:

- exact duplicate bytes, based on SHA-256, suggests the archived image content is identical;
- approximate visual similarity, based on a small perceptual thumbnail hash, suggests images may look similar but is not proof of duplicate experimental evidence.

A duplicate image is not automatically an error. It may be a repeated export, a deliberate replicate, or a naming issue. Phase 6 records it for review rather than deleting evidence.

In [21]:
# Duplicate image diagnostics.

readable_images = image_integrity_properties.query("readable == True").copy()

duplicate_records = []
if "sha256" in readable_images.columns:
    sha_counts = readable_images["sha256"].dropna().value_counts()
    for sha, count in sha_counts[sha_counts > 1].items():
        paths = readable_images.loc[readable_images["sha256"] == sha, "archive_path"].tolist()
        duplicate_records.append({
            "duplicate_type": "exact_sha256_duplicate",
            "duplicate_key": sha,
            "duplicate_count": int(count),
            "archive_paths": " | ".join(paths),
            "interpretation": "Exact byte-level duplicate; review source context before treating as duplicate experimental evidence.",
        })

if "thumbnail_dhash" in readable_images.columns:
    dh_counts = readable_images["thumbnail_dhash"].dropna().value_counts()
    for dh, count in dh_counts[dh_counts > 1].items():
        paths = readable_images.loc[readable_images["thumbnail_dhash"] == dh, "archive_path"].head(20).tolist()
        duplicate_records.append({
            "duplicate_type": "thumbnail_dhash_duplicate_or_near_duplicate",
            "duplicate_key": dh,
            "duplicate_count": int(count),
            "archive_paths": " | ".join(paths),
            "interpretation": "Small-thumbnail visual hash collision or near duplicate; use only as a review cue.",
        })

duplicate_image_diagnostics = pd.DataFrame(duplicate_records)
write_csv(duplicate_image_diagnostics, OUTPUT_DIR / "phase6_duplicate_image_diagnostics.csv")

print("Duplicate diagnostic groups:", len(duplicate_image_diagnostics))
display_frame("Duplicate Diagnostics", duplicate_image_diagnostics, n=10)

Duplicate diagnostic groups: 2


### Duplicate Diagnostics

,duplicate_type,duplicate_key,duplicate_count,archive_paths,interpretation
0,thumbnail_dhash_duplicate_or_near_duplicate,004896333307cc08,2,images/resized_IMG_8429.png | images/resized_IMG_8664.png,Small-thumbnail visual hash collision or near duplicate; use only as a review cue.
1,thumbnail_dhash_duplicate_or_near_duplicate,02489633330fcc28,2,images/resized_IMG_8444.png | images/resized_IMG_8636.png,Small-thumbnail visual hash collision or near duplicate; use only as a review cue.


In [22]:
# Combine archive linkage with computed image properties for metadata-group summaries.

linkage_single = image_index_to_archive_linkage.query("linkage_status == 'matched_one_file'").copy()
index_with_files = working_index.merge(
    linkage_single[["source_csv_record", "primary_archive_path", "match_method", "linkage_status"]],
    on="source_csv_record",
    how="left",
)
index_with_files = index_with_files.merge(
    image_integrity_properties,
    left_on="primary_archive_path",
    right_on="archive_path",
    how="left",
)

write_csv(index_with_files, OUTPUT_DIR / "phase6_image_metadata_feature_layer.csv")

metadata_completeness = image_index_profile.copy()
metadata_completeness["phase6_interpretation"] = np.where(
    metadata_completeness["missing"] > 0,
    "Missing values retained as missing source metadata; do not impute as experimental measurements.",
    "Field complete in supplied image_index.csv."
)
write_csv(metadata_completeness, OUTPUT_DIR / "phase6_metadata_completeness.csv")

# Group summaries are descriptive only. They do not validate causal relationships.
group_summary_records = []
for group_col in ["ink_type_ph", "shape", "intensity", "completeness_sam", "contain_sps"]:
    if group_col in index_with_files.columns and "gray_mean" in index_with_files.columns:
        summary = index_with_files.groupby(group_col, dropna=False).agg(
            records=("source_csv_record", "count"),
            matched_images=("primary_archive_path", lambda x: int(pd.Series(x).notna().sum())),
            readable_images=("readable", lambda x: int(pd.Series(x).fillna(False).sum())),
            gray_mean_median=("gray_mean", "median"),
            gray_std_median=("gray_std", "median"),
            foreground_area_fraction_median=("foreground_area_fraction", "median"),
            edge_density_simple_median=("edge_density_simple", "median"),
        ).reset_index()
        summary.insert(0, "group_field", group_col)
        summary = summary.rename(columns={group_col: "group_value"})
        group_summary_records.append(summary)

if group_summary_records:
    phase6_metadata_group_feature_summary = pd.concat(group_summary_records, ignore_index=True)
else:
    phase6_metadata_group_feature_summary = pd.DataFrame()

write_csv(phase6_metadata_group_feature_summary, OUTPUT_DIR / "phase6_metadata_group_feature_summary.csv")
display_frame("Metadata and Feature Layer Preview", index_with_files, n=8)

### Metadata and Feature Layer Preview

,image_id,date,ink_type_ph,shape,intensity,completeness_sam,contain_sps,source_csv_record,image_id_norm,image_id_stem_norm,expected_filename,primary_archive_path,match_method,linkage_status,archive_path,readable,sha256,image_format,width_px,height_px,pixel_count,mode,channel_count,feature_image_width_px,feature_image_height_px,gray_mean,gray_std,gray_min,gray_p05,gray_median,gray_p95,gray_max,red_mean,green_mean,blue_mean,background_gray_estimate,foreground_threshold,foreground_method,foreground_area_fraction,otsu_threshold,otsu_method,edge_density_simple,thumbnail_dhash,blank_like_flag,low_contrast_flag,very_dark_flag,very_bright_flag,feature_interpretation_boundary,file_name,suffix,file_size_bytes,crc,zip_modified_datetime
0,7586,2025/05/28,ALG2.0,grid,1,NaN,NaN,1,7586,7586,resized_IMG_7586.png,images/resized_IMG_7586.png,phase4_expected_filename_exact_file_name,matched_one_file,images/resized_IMG_7586.png,True,eae137ce228132f2ca003622d445194779ad5a243e17c0cfe2dc796d29f1b2b5,PNG,800,800,640000,RGB,3,512,512,0.650141,0.029879,0.550218,0.599353,0.656672,0.690725,0.722071,0.667320,0.664701,0.455325,0.668894,0.08,border_difference_threshold,0.019421,0.646549,numpy_histogram_otsu,0.139950,494b455d45495515,False,True,False,False,Computed image descriptors support exploratory audit only; they are not validated printability or morphology labels.,resized_IMG_7586.png,.png,521825,a4da8956,2025-10-26 20:52:26
1,7587,2025/05/28,ALG2.0,grid,1,NaN,NaN,2,7587,7587,resized_IMG_7587.png,images/resized_IMG_7587.png,phase4_expected_filename_exact_file_name,matched_one_file,images/resized_IMG_7587.png,True,4d0743dd66e96d34fa453d2ffa9abf7a35a69a97bf7e4aad643c2c0d204c50d1,PNG,800,800,640000,RGB,3,512,512,0.691304,0.031327,0.602933,0.637766,0.697576,0.733878,0.770231,0.701391,0.707789,0.498300,0.711784,0.08,border_difference_threshold,0.029190,0.687562,numpy_histogram_otsu,0.139736,4949454545455515,False,True,False,False,Computed image descriptors support exploratory audit only; they are not validated printability or morphology labels.,resized_IMG_7587.png,.png,497828,a6d02c80,2025-08-18 13:14:32
2,7588,2025/05/28,ALG2.0,grid,1,NaN,NaN,3,7588,7588,resized_IMG_7588.png,images/resized_IMG_7588.png,phase4_expected_filename_exact_file_name,matched_one_file,images/resized_IMG_7588.png,True,80e3aa3838bc5aebf7687b47be317e7a8f4d1cceb778594c2dc978258be28135,PNG,800,800,640000,RGB,3,512,512,0.700749,0.030867,0.380821,0.647832,0.705325,0.744793,0.778078,0.713389,0.716949,0.503053,0.717373,0.08,border_difference_threshold,0.018761,0.696610,numpy_histogram_otsu,0.135166,4949414549495505,False,True,False,False,Computed image descriptors support exploratory audit only; they are not validated printability or morphology labels.,resized_IMG_7588.png,.png,493173,cf3fd4fe,2025-08-18 13:14:32
3,7589,2025/05/28,ALG2.0,grid,50,NaN,NaN,4,7589,7589,resized_IMG_7589.png,images/resized_IMG_7589.png,phase4_expected_filename_exact_file_name,matched_one_file,images/resized_IMG_7589.png,True,6824147cbb36ea513276699cdee82c57be3ee3fcf1630fa00e617e9ca99ba41d,PNG,800,800,640000,RGB,3,512,512,0.671649,0.040423,0.471932,0.607747,0.684580,0.722931,0.765727,0.691706,0.685662,0.473779,0.692424,0.08,border_difference_threshold,0.072269,0.664161,numpy_histogram_otsu,0.131649,4949494549095551,False,True,False,False,Computed image descriptors support exploratory audit only; they are not validated printability or morphology labels.,resized_IMG_7589.png,.png,514026,72cc3f33,2025-08-18 13:14:32
4,7590,2025/05/28,ALG2.0,grid,50,no,NaN,5,7590,7590,resized_IMG_7590.png,images/resized_IMG_7590.png,phase4_expected_filename_exact_file_name,matched_one_file,images/resized_IMG_7590.png,True,30a698b99dbeaab2b7f8f42cfe828f9985c906b1996328ee036abec188397d54,PNG,800,800,640000,RGB,3,512,512,0.687742,0.037148,0.513776,0.625660,0.697781,0.735314,0.766073,0.693370,0.704579,0.504394,0.706741,0.08,border_difference_threshold,0.053719,0.679839,numpy_histogram_otsu,0.127602,6949215925457511,False,True,False,Fals

## Visualisation Principles

The figures below are designed for scientific review, not decoration. Each figure should answer a concrete evidence question:

- What image files exist?
- How well do source metadata records match actual files?
- Are dimensions, formats or channels consistent?
- Are metadata fields missing?
- Do simple image descriptors vary across recorded categories such as ink type, printed shape or irradiation intensity?
- Which images deserve manual review before any downstream modelling?

The notebook uses two levels of visualisation:

1. **Audit figures** summarise the archive, linkage and metadata completeness.
2. **Review panels** show small groups of real images so a human can inspect whether the numeric signals make scientific sense.

Interpretation remains cautious. If one shape category has a higher foreground-area fraction, that does not mean the shape has better printability. It only means that the current image-processing descriptor differs under the notebook's current assumptions. The polymer-science interpretation still needs formulation context, rheology context and source-supported experimental labels.


In [24]:
# Scientific visualisations.

figure_register = []


def boxplot_with_category_labels(ax, data, labels, **kwargs):
    """Use the Matplotlib keyword supported by the local version."""
    try:
        return ax.boxplot(data, tick_labels=labels, **kwargs)
    except TypeError:
        return ax.boxplot(data, labels=labels, **kwargs)


# 1. Image file-format counts.
if not image_file_inventory.empty:
    counts = image_file_inventory["suffix"].value_counts().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(6.5, 4))
    counts.plot(kind="bar", ax=ax, color="#4C78A8")
    ax.set_title("Phase 6 Image Archive File-format Counts")
    ax.set_xlabel("File extension")
    ax.set_ylabel("Image file count")
    ax.grid(axis="y", alpha=0.25)
    save_fig(fig, "phase6_01_image_file_format_counts.png", "Image archive file-format counts", f"{len(image_file_inventory)} image candidate files", "ZIP inventory grouped by file suffix", "File suffix is a storage format cue, not a scientific measurement.", figure_register)

# 2. Image-index to archive linkage coverage.
if not image_index_to_archive_linkage.empty:
    counts = image_index_to_archive_linkage["linkage_status"].value_counts().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(7, 4))
    counts.plot(kind="bar", ax=ax, color="#59A14F")
    ax.set_title("Image-index to Archive Matching Coverage")
    ax.set_xlabel("Linkage status")
    ax.set_ylabel("Image-index records")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.25)
    save_fig(fig, "phase6_02_image_index_matching_coverage.png", "Image-index to archive matching coverage", f"{len(image_index_to_archive_linkage)} image-index records", "Metadata-to-file matching using documented filename rules", "File matching does not prove physical-sample identity.", figure_register)

# 3. Image dimensions scatter.
readable_for_plot = image_integrity_properties.query("readable == True").copy()
if not readable_for_plot.empty and {"width_px", "height_px"}.issubset(readable_for_plot.columns):
    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.scatter(readable_for_plot["width_px"], readable_for_plot["height_px"], s=18, alpha=0.65, color="#F28E2B")
    ax.set_title("Readable Image Pixel Dimensions")
    ax.set_xlabel("Width (pixels)")
    ax.set_ylabel("Height (pixels)")
    ax.grid(alpha=0.25)
    save_fig(fig, "phase6_03_image_dimensions_scatter.png", "Readable image pixel dimensions", f"{len(readable_for_plot)} readable images", "Pillow image metadata extraction", "Pixel dimensions describe digital files, not physical print dimensions unless calibration metadata is available.", figure_register)

# 4. Metadata missingness heatmap.
missing_cols = [c for c in ["image_id", "date", "ink_type_ph", "shape", "intensity", "completeness_sam", "contain_sps"] if c in image_index.columns]
if missing_cols:
    missing_matrix = image_index[missing_cols].isna().astype(int).to_numpy()
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.imshow(missing_matrix, aspect="auto", interpolation="nearest", cmap="Greys")
    ax.set_title("Image Metadata Missingness Map")
    ax.set_xlabel("Metadata field")
    ax.set_ylabel("Image-index row")
    ax.set_xticks(range(len(missing_cols)))
    ax.set_xticklabels(missing_cols, rotation=35, ha="right")
    save_fig(fig, "phase6_04_metadata_missingness_map.png", "Image metadata missingness map", f"{len(image_index)} image-index records", "Binary missingness display across selected metadata fields", "Missingness records source availability only; missing values are not imputed.", figure_register)

# 5. Ink-type distribution.
for col, filename, title in [
    ("ink_type_ph", "phase6_05_ink_type_distribution.png", "Ink-type Metadata Distribution"),
    ("shape", "phase6_06_printed_shape_distribution.png", "Printed-shape Metadata Distribution"),
    ("intensity", "phase6_07_irradiation_intensity_distribution.png", "Irradiation-intensity Metadata Distribution"),
]:
    if col in image_index.columns:
        counts = image_index[col].fillna("<missing>").astype(str).value_counts().head(25)
        fig, ax = plt.subplots(figsize=(8, max(4, 0.25 * len(counts))))
        counts.sort_values().plot(kind="barh", ax=ax, color="#4C78A8")
        ax.set_title(title)
        ax.set_xlabel("Image-index records")
        ax.set_ylabel(col)
        ax.grid(axis="x", alpha=0.25)
        save_fig(fig, filename, title, f"{len(image_index)} image-index records", f"Value counts for image_index.{col}", "Metadata categories describe source labels; they are not automatically standardised experimental variables.", figure_register)

# 8. Brightness versus contrast.
if not readable_for_plot.empty and {"gray_mean", "gray_std"}.issubset(readable_for_plot.columns):
    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.scatter(readable_for_plot["gray_mean"], readable_for_plot["gray_std"], s=18, alpha=0.6, color="#B07AA1")
    ax.set_title("Image Brightness and Contrast Screening")
    ax.set_xlabel("Mean grayscale intensity (0 to 1)")
    ax.set_ylabel("Grayscale standard deviation")
    ax.grid(alpha=0.25)
    save_fig(fig, "phase6_08_brightness_contrast_scatter.png", "Image brightness and contrast screening", f"{len(readable_for_plot)} readable images", "Thumbnail grayscale intensity summary", "Brightness and contrast may reflect imaging conditions as well as printed material appearance.", figure_register)

# 9. Foreground area by shape.
if "shape" in index_with_files.columns and "foreground_area_fraction" in index_with_files.columns:
    temp = index_with_files.dropna(subset=["foreground_area_fraction"]).copy()
    if not temp.empty:
        order = temp["shape"].fillna("<missing>").astype(str).value_counts().head(12).index.tolist()
        data = [temp.loc[temp["shape"].fillna("<missing>").astype(str) == cat, "foreground_area_fraction"].dropna().values for cat in order]
        fig, ax = plt.subplots(figsize=(8, 4.5))
        boxplot_with_category_labels(ax, data, order, showfliers=False)
        ax.set_title("Foreground-area Fraction by Recorded Shape Category")
        ax.set_xlabel("Recorded shape category")
        ax.set_ylabel("Foreground-area fraction")
        ax.tick_params(axis="x", rotation=30)
        ax.grid(axis="y", alpha=0.25)
        save_fig(fig, "phase6_09_foreground_area_by_shape.png", "Foreground-area fraction by recorded shape category", f"{len(temp)} linked readable images", "Border-background thresholding on thumbnail images", "Foreground area is a computed descriptor and not a validated printability score.", figure_register)

# 10. Example image grid.
def load_rgb_preview_from_zip(zf: zipfile.ZipFile, archive_path: str, max_dim: int) -> Image.Image:
    with zf.open(archive_path, "r") as f:
        data = f.read()
    with Image.open(BytesIO(data)) as img:
        return pil_to_rgb_thumbnail(img, max_dim).copy()

sample_paths = readable_for_plot["archive_path"].dropna().head(CONFIG["example_grid_limit"]).tolist()
if sample_paths:
    cols = 4
    rows = math.ceil(len(sample_paths) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(10, 2.8 * rows))
    axes = np.array(axes).reshape(-1)
    with zipfile.ZipFile(IMAGE_ZIP_PATH, "r") as zf:
        for ax, archive_path in zip(axes, sample_paths):
            img = load_rgb_preview_from_zip(zf, archive_path, CONFIG["max_preview_dimension_px"])
            ax.imshow(img)
            ax.set_title(Path(archive_path).name[:32], fontsize=7)
            ax.axis("off")
    for ax in axes[len(sample_paths):]:
        ax.axis("off")
    save_fig(fig, "phase6_10_example_image_grid.png", "Example source image grid", f"{len(sample_paths)} readable source images", "First readable images loaded from the ZIP archive and displayed as thumbnails", "The grid supports visual audit only and is not a representative statistical sample unless sampling is defined.", figure_register)

# 11. Source versus processed comparison.
comparison_paths = readable_for_plot["archive_path"].dropna().head(CONFIG["processed_comparison_limit"]).tolist()
if comparison_paths:
    fig, axes = plt.subplots(len(comparison_paths), 3, figsize=(9, 3 * len(comparison_paths)))
    axes = np.array(axes).reshape(len(comparison_paths), 3)
    with zipfile.ZipFile(IMAGE_ZIP_PATH, "r") as zf:
        for row_idx, archive_path in enumerate(comparison_paths):
            img = load_rgb_preview_from_zip(zf, archive_path, CONFIG["max_preview_dimension_px"])
            arr = np.asarray(img)
            gray = rgb_array_to_gray(arr)
            mask, bg, threshold, method = foreground_mask_from_border(gray)
            axes[row_idx, 0].imshow(img)
            axes[row_idx, 0].set_title("Source thumbnail", fontsize=8)
            axes[row_idx, 1].imshow(gray, cmap="gray")
            axes[row_idx, 1].set_title("Grayscale", fontsize=8)
            axes[row_idx, 2].imshow(mask, cmap="gray")
            axes[row_idx, 2].set_title("Foreground mask", fontsize=8)
            for ax in axes[row_idx, :]:
                ax.axis("off")
            axes[row_idx, 0].set_ylabel(Path(archive_path).name[:18], fontsize=7)
    save_fig(fig, "phase6_11_source_processed_comparison.png", "Source versus processed image comparison", f"{len(comparison_paths)} readable source images", "Thumbnail display, grayscale conversion, and border-background foreground mask", "Preprocessing can change interpretation; the mask is a review tool, not a ground-truth segmentation.", figure_register)

figure_register_df = pd.DataFrame(figure_register)
write_csv(figure_register_df, OUTPUT_DIR / "phase6_figure_register.csv")
display_frame("Phase 6 Figure Register", figure_register_df, n=30)

### Phase 6 Figure Register

,figure_file,title,records_analysed,method,scientific_interpretation,limitations,relative_path
0,phase6_01_image_file_format_counts.png,Image archive file-format counts,1031 image candidate files,ZIP inventory grouped by file suffix,Image archive file-format counts,"File suffix is a storage format cue, not a scientific measurement.",data\processed\image_file_audit_optional\figures\phase6_01_image_file_format_counts.png
1,phase6_02_image_index_matching_coverage.png,Image-index to archive matching coverage,1031 image-index records,Metadata-to-file matching using documented filename rules,Image-index to archive matching coverage,File matching does not prove physical-sample identity.,data\processed\image_file_audit_optional\figures\phase6_02_image_index_matching_coverage.png
2,phase6_03_image_dimensions_scatter.png,Readable image pixel dimensions,1031 readable images,Pillow image metadata extraction,Readable image pixel dimensions,"Pixel dimensions describe digital files, not physical print dimensions unless calibration metadata is available.",data\processed\image_file_audit_optional\figures\phase6_03_image_dimensions_scatter.png
3,phase6_04_metadata_missingness_map.png,Image metadata missingness map,1031 image-index records,Binary missingness display across selected metadata fields,Image metadata missingness map,Missingness records source availability only; missing values are not imputed.,data\processed\image_file_audit_optional\figures\phase6_04_metadata_missingness_map.png
4,phase6_05_ink_type_distribution.png,Ink-type Metadata Distribution,1031 image-index records,Value counts for image_index.ink_type_ph,Ink-type Metadata Distribution,Metadata categories describe source labels; they are not automatically standardised experimental variables.,data\processed\image_file_audit_optional\figures\phase6_05_ink_type_distribution.png
5,phase6_06_printed_shape_distribution.png,Printed-shape Metadata Distribution,1031 image-index records,Value counts for image_index.shape,Printed-shape Metadata Distribution,Metadata categories describe source labels; they are not automatically standardised experimental variables.,data\processed\image_file_audit_optional\figures\phase6_06_printed_shape_distribution.png
6,phase6_07_irradiation_intensity_distribution.png,Irradiation-intensity Metadata Distribution,1031 image-index records,Value counts for image_index.intensity,Irradiation-intensity Metadata Distribution,Metadata categories describe source labels; they are not automatically standardised experimental variables.,data\processed\image_file_audit_optional\figures\phase6_07_irradiation_intensity_distribution.png
7,phase6_08_brightness_contrast_scatter.png,Image brightness and contrast screening,1031 readable images,Thumbnail grayscale intensity summary,Image brightness and contrast screening,Brightness and contrast may reflect imaging conditions as well as printed material appearance.,data\processed\image_file_audit_optional\figures\phase6_08_brightness_contrast_scatter.png
8,phase6_09_foreground_area_by_shape.png,Foreground-area fraction by recorded shape category,1031 linked readable images,Border-background thresholding on thumbnail images,Foreground-area fraction by recorded shape category,Foreground area is a computed descriptor and not a validated printability score.,data\processed\image_file_audit_optional\figures\phase6_09_foreground_area_by_shape.png
9,phase6_10_example_image_grid.png,Example source image grid,12 readable source images,First readable images loaded from the ZIP archive and displayed as thumbnails,Example source image grid,The grid supports visual audit only and is not a representative statistical sample unless sampling is defined.,data\processed\image_file_audit_optional\figures\phase6_10_example_image_grid.png


## Second-round Visual Review Layer

This section adds richer visualisation using the actual Phase 6 outputs created above.

The goal is not to make the notebook look prettier. The goal is to make the evidence easier to audit:

- example image grids grouped by ink type, shape and irradiation intensity;
- feature heatmaps showing how simple pixel descriptors differ across metadata groups;
- panels for rows with missing metadata;
- panels for duplicate or near-duplicate image groups;
- panels for unusual images such as very bright, very dark, low-contrast or outlying feature values.

This is where data science and polymer science meet in a practical way. Data science helps compress many images into reviewable summaries. Polymer science reminds us what those summaries can and cannot mean physically. A descriptor can suggest “please review this image”, but it should not silently become “this hydrogel printed well”.


In [26]:
# Second-round visual review panels and richer exploratory figures.
# These cells are deliberately CPU-only and memory-conscious. Images are still loaded
# from images.zip one at a time, only as thumbnails, and only for selected panels.

PANEL_MAX_GROUPS = 6
PANEL_SAMPLES_PER_GROUP = 3
PANEL_MAX_IMAGES = 18
PANEL_PREVIEW_DIM = 320

visual_review_panel_records = []
visual_group_feature_summary_records = []


def safe_display_value(value) -> str:
    if pd.isna(value):
        return "<missing>"
    text = str(value).strip()
    return text if text else "<blank>"


def slugify_for_filename(value, max_len: int = 48) -> str:
    text = safe_display_value(value).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")
    return (text[:max_len].strip("_") or "missing")


def row_lookup(row, column, default=""):
    if column not in row.index:
        return default
    value = row[column]
    if pd.isna(value):
        return default
    return value


def panel_base_records() -> pd.DataFrame:
    df = index_with_files.copy()
    if df.empty:
        return df
    if "primary_archive_path" in df.columns:
        df["panel_archive_path"] = df["primary_archive_path"]
    else:
        df["panel_archive_path"] = np.nan
    if "archive_path" in df.columns:
        df["panel_archive_path"] = df["panel_archive_path"].fillna(df["archive_path"])
    if "readable" in df.columns:
        readable_mask = df["readable"].fillna(False).astype(bool)
    else:
        readable_mask = pd.Series(False, index=df.index)
    df = df.loc[readable_mask & df["panel_archive_path"].notna()].copy()
    sort_cols = [c for c in ["source_csv_record", "image_id", "panel_archive_path"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols, kind="mergesort")
    return df


linked_readable_for_panels = panel_base_records()


def record_panel_item(panel_type: str, row: pd.Series, reason: str, panel_field: str = "", panel_value: str = "") -> None:
    visual_review_panel_records.append({
        "panel_type": panel_type,
        "panel_field": panel_field,
        "panel_value": panel_value,
        "source_csv_record": row_lookup(row, "source_csv_record", ""),
        "image_id": row_lookup(row, "image_id", ""),
        "archive_path": row_lookup(row, "panel_archive_path", row_lookup(row, "archive_path", "")),
        "reason_for_review": reason,
        "scientific_boundary": "Review panel only; visual appearance is not a validated printability, viability or clinical-performance label.",
    })


def image_panel_title(row: pd.Series, extra_columns: list[str] | None = None) -> str:
    parts = []
    image_id = row_lookup(row, "image_id", "")
    if image_id != "":
        parts.append(str(image_id)[:22])
    path = row_lookup(row, "panel_archive_path", row_lookup(row, "archive_path", ""))
    if path != "":
        parts.append(Path(str(path)).name[:28])
    for col in extra_columns or []:
        if col in row.index:
            parts.append(f"{col}={safe_display_value(row[col])[:20]}")
    return "\n".join(parts[:3])


def make_image_grid_from_rows(panel_df: pd.DataFrame, filename: str, title: str, records_analysed: str, method: str, limitation: str, label_columns: list[str] | None = None, max_images: int = PANEL_MAX_IMAGES) -> bool:
    if panel_df.empty or "panel_archive_path" not in panel_df.columns:
        return False
    plot_df = panel_df.dropna(subset=["panel_archive_path"]).head(max_images).copy()
    if plot_df.empty:
        return False

    n = len(plot_df)
    cols = min(3 if n <= 9 else 4, n)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.9, rows * 3.1))
    axes = np.array(axes).reshape(-1)

    with zipfile.ZipFile(IMAGE_ZIP_PATH, "r") as zf:
        for ax, (_, row) in zip(axes, plot_df.iterrows()):
            archive_path = str(row["panel_archive_path"])
            try:
                img = load_rgb_preview_from_zip(zf, archive_path, PANEL_PREVIEW_DIM)
                ax.imshow(img)
            except Exception as exc:
                ax.text(0.5, 0.5, f"Could not load\n{Path(archive_path).name}\n{type(exc).__name__}", ha="center", va="center", fontsize=7)
            ax.set_title(image_panel_title(row, label_columns), fontsize=7)
            ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    save_fig(fig, filename, title, records_analysed, method, limitation, figure_register)
    return True


def make_category_example_grid(group_col: str, filename: str, title: str) -> None:
    if linked_readable_for_panels.empty or group_col not in linked_readable_for_panels.columns:
        return

    df = linked_readable_for_panels.copy()
    df["__panel_group_value"] = df[group_col].apply(safe_display_value)
    top_groups = df["__panel_group_value"].value_counts().head(PANEL_MAX_GROUPS).index.tolist()

    selected_frames = []
    for group_value in top_groups:
        group_rows = df.loc[df["__panel_group_value"] == group_value].head(PANEL_SAMPLES_PER_GROUP).copy()
        if not group_rows.empty:
            for _, row in group_rows.iterrows():
                record_panel_item(
                    panel_type="category_example_grid",
                    row=row,
                    reason=f"Representative thumbnail selected from metadata group {group_col}={group_value}",
                    panel_field=group_col,
                    panel_value=group_value,
                )
            selected_frames.append(group_rows)

    if not selected_frames:
        return

    panel_df = pd.concat(selected_frames, ignore_index=True)
    make_image_grid_from_rows(
        panel_df,
        filename,
        title,
        f"{len(panel_df)} thumbnails from {len(top_groups)} {group_col} groups",
        f"Deterministic first-{PANEL_SAMPLES_PER_GROUP} sampling from the most frequent linked readable {group_col} groups",
        "Thumbnails are examples for audit only; group appearance may be affected by lighting, cropping, scale and source-image selection.",
        label_columns=[group_col],
    )


for group_col, filename, title in [
    ("ink_type_ph", "phase6_12_example_grid_by_ink_type.png", "Example Image Grid by Recorded Ink Type"),
    ("shape", "phase6_13_example_grid_by_shape.png", "Example Image Grid by Recorded Shape"),
    ("intensity", "phase6_14_example_grid_by_irradiation_intensity.png", "Example Image Grid by Recorded Irradiation Intensity"),
]:
    make_category_example_grid(group_col, filename, title)


# Feature heatmaps by metadata group.
feature_cols = [
    "gray_mean",
    "gray_std",
    "foreground_area_fraction",
    "edge_density_simple",
    "width_px",
    "height_px",
]
feature_cols = [c for c in feature_cols if c in linked_readable_for_panels.columns]


def make_group_feature_heatmap(group_col: str, filename: str, title: str) -> None:
    if linked_readable_for_panels.empty or group_col not in linked_readable_for_panels.columns or not feature_cols:
        return

    temp = linked_readable_for_panels.copy()
    temp["__group_value"] = temp[group_col].apply(safe_display_value)
    top_groups = temp["__group_value"].value_counts().head(PANEL_MAX_GROUPS).index.tolist()
    temp = temp.loc[temp["__group_value"].isin(top_groups)].copy()
    if temp.empty:
        return

    summary = temp.groupby("__group_value", dropna=False).agg(
        records=("source_csv_record", "count"),
        **{f"{col}_median": (col, "median") for col in feature_cols}
    ).reset_index().rename(columns={"__group_value": "group_value"})
    summary.insert(0, "group_field", group_col)
    for row in summary.to_dict(orient="records"):
        visual_group_feature_summary_records.append(row)

    heat_cols = [f"{col}_median" for col in feature_cols]
    heat_data = summary[heat_cols].astype(float).to_numpy()
    scaled = np.zeros_like(heat_data, dtype=float)
    for idx in range(heat_data.shape[1]):
        col = heat_data[:, idx]
        finite = np.isfinite(col)
        if finite.sum() == 0:
            scaled[:, idx] = np.nan
            continue
        c_min = np.nanmin(col)
        c_max = np.nanmax(col)
        if not c_max > c_min:
            scaled[:, idx] = 0.5
        else:
            scaled[:, idx] = (col - c_min) / (c_max - c_min)

    fig, ax = plt.subplots(figsize=(8.5, max(3.8, 0.48 * len(summary))))
    im = ax.imshow(scaled, aspect="auto", interpolation="nearest", cmap="viridis", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("Image descriptor, scaled within this figure")
    ax.set_ylabel(group_col)
    ax.set_xticks(range(len(heat_cols)))
    ax.set_xticklabels([c.replace("_median", "") for c in heat_cols], rotation=35, ha="right")
    ax.set_yticks(range(len(summary)))
    ax.set_yticklabels([f"{v} (n={n})" for v, n in zip(summary["group_value"], summary["records"])])
    fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03, label="Scaled median descriptor")
    save_fig(
        fig,
        filename,
        title,
        f"{len(temp)} linked readable images across {len(summary)} {group_col} groups",
        "Median image descriptors by metadata group, min-max scaled per descriptor for visual comparison",
        "Scaled heatmap shows descriptor patterns only; it does not establish causality, printability or biological performance.",
        figure_register,
    )


for group_col, filename, title in [
    ("ink_type_ph", "phase6_15_feature_heatmap_by_ink_type.png", "Image Feature Heatmap by Recorded Ink Type"),
    ("shape", "phase6_16_feature_heatmap_by_shape.png", "Image Feature Heatmap by Recorded Shape"),
    ("intensity", "phase6_17_feature_heatmap_by_irradiation_intensity.png", "Image Feature Heatmap by Recorded Irradiation Intensity"),
]:
    make_group_feature_heatmap(group_col, filename, title)

phase6_visual_group_feature_summary = pd.DataFrame(visual_group_feature_summary_records)
write_csv(phase6_visual_group_feature_summary, OUTPUT_DIR / "phase6_visual_group_feature_summary.csv")


# Metadata missingness review panel.
review_missing_cols = [c for c in ["image_id", "date", "ink_type_ph", "shape", "intensity", "completeness_sam", "contain_sps"] if c in index_with_files.columns]
if review_missing_cols:
    missing_temp = index_with_files.copy()
    missing_temp["missing_metadata_count"] = missing_temp[review_missing_cols].isna().sum(axis=1)
    missing_temp["missing_metadata_fields"] = missing_temp[review_missing_cols].isna().apply(lambda row: " | ".join(row.index[row].tolist()), axis=1)
    phase6_missing_metadata_review_panel = missing_temp.loc[missing_temp["missing_metadata_count"] > 0].copy()
else:
    phase6_missing_metadata_review_panel = pd.DataFrame(columns=["missing_metadata_count", "missing_metadata_fields"])

write_csv(phase6_missing_metadata_review_panel, OUTPUT_DIR / "phase6_missing_metadata_review_panel.csv")

if not phase6_missing_metadata_review_panel.empty and not linked_readable_for_panels.empty:
    missing_with_images = linked_readable_for_panels.merge(
        phase6_missing_metadata_review_panel[["source_csv_record", "missing_metadata_count", "missing_metadata_fields"]],
        on="source_csv_record",
        how="inner",
    ).sort_values(["missing_metadata_count", "source_csv_record"], ascending=[False, True])
    for _, row in missing_with_images.head(PANEL_MAX_IMAGES).iterrows():
        record_panel_item(
            panel_type="missing_metadata_review_panel",
            row=row,
            reason=f"Missing metadata fields: {row_lookup(row, 'missing_metadata_fields', '')}",
            panel_field="missing_metadata_fields",
            panel_value=row_lookup(row, "missing_metadata_fields", ""),
        )
    make_image_grid_from_rows(
        missing_with_images,
        "phase6_18_missing_metadata_image_review_panel.png",
        "Image Review Panel for Rows with Missing Metadata",
        f"{min(len(missing_with_images), PANEL_MAX_IMAGES)} linked readable images with missing metadata",
        "Rows ranked by count of missing selected metadata fields",
        "Missing metadata is source evidence, not a reason to invent labels or drop records automatically.",
        label_columns=["missing_metadata_fields"],
    )


# Duplicate-group review panel.
duplicate_panel_rows = []
if not duplicate_image_diagnostics.empty:
    path_to_rows = linked_readable_for_panels.copy()
    for group_number, dup_row in enumerate(duplicate_image_diagnostics.head(PANEL_MAX_GROUPS).itertuples(index=False), start=1):
        archive_paths = str(getattr(dup_row, "archive_paths", "")).split(" | ")
        for archive_path in archive_paths[:PANEL_SAMPLES_PER_GROUP]:
            archive_path = archive_path.strip()
            if not archive_path:
                continue
            matched = path_to_rows.loc[path_to_rows["panel_archive_path"] == archive_path].copy()
            if matched.empty:
                matched = pd.DataFrame([{"panel_archive_path": archive_path, "image_id": "", "source_csv_record": ""}])
            matched = matched.head(1).copy()
            matched["duplicate_group_number"] = group_number
            matched["duplicate_type"] = getattr(dup_row, "duplicate_type", "")
            matched["duplicate_key"] = getattr(dup_row, "duplicate_key", "")
            duplicate_panel_rows.append(matched)

if duplicate_panel_rows:
    phase6_duplicate_group_review_panel = pd.concat(duplicate_panel_rows, ignore_index=True)
else:
    phase6_duplicate_group_review_panel = pd.DataFrame(columns=["duplicate_group_number", "duplicate_type", "duplicate_key", "panel_archive_path"])

write_csv(phase6_duplicate_group_review_panel, OUTPUT_DIR / "phase6_duplicate_group_review_panel.csv")

if not phase6_duplicate_group_review_panel.empty:
    for _, row in phase6_duplicate_group_review_panel.head(PANEL_MAX_IMAGES).iterrows():
        record_panel_item(
            panel_type="duplicate_group_review_panel",
            row=row,
            reason=f"Duplicate cue group {row_lookup(row, 'duplicate_group_number', '')}: {row_lookup(row, 'duplicate_type', '')}",
            panel_field="duplicate_type",
            panel_value=row_lookup(row, "duplicate_type", ""),
        )
    make_image_grid_from_rows(
        phase6_duplicate_group_review_panel,
        "phase6_19_duplicate_group_review_panel.png",
        "Duplicate and Near-duplicate Image Review Panel",
        f"{min(len(phase6_duplicate_group_review_panel), PANEL_MAX_IMAGES)} images from duplicate diagnostic groups",
        "Exact SHA-256 and thumbnail-hash duplicate cues displayed as thumbnails",
        "Duplicate cues require human review; visual hash matches may be repeated images, similar images or hash collisions.",
        label_columns=["duplicate_group_number", "duplicate_type"],
    )


# Unusual-image review panel.
def robust_abs_z(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    median = values.median(skipna=True)
    mad = (values - median).abs().median(skipna=True)
    if pd.notna(mad) and mad > 0:
        return ((values - median).abs() / (1.4826 * mad)).replace([np.inf, -np.inf], np.nan)
    std = values.std(skipna=True)
    if pd.notna(std) and std > 0:
        return ((values - median).abs() / std).replace([np.inf, -np.inf], np.nan)
    return pd.Series(np.zeros(len(values)), index=values.index, dtype=float)

unusual_cols = [c for c in ["gray_mean", "gray_std", "foreground_area_fraction", "edge_density_simple"] if c in linked_readable_for_panels.columns]
unusual_temp = linked_readable_for_panels.copy()
for col in unusual_cols:
    unusual_temp[f"{col}_robust_abs_z"] = robust_abs_z(unusual_temp[col])

flag_cols = [c for c in ["blank_like_flag", "low_contrast_flag", "very_dark_flag", "very_bright_flag"] if c in unusual_temp.columns]

unusual_records = []
for idx, row in unusual_temp.iterrows():
    reasons = []
    scores = []
    for col in flag_cols:
        if bool(row_lookup(row, col, False)):
            reasons.append(col)
            scores.append(3.0)
    for col in unusual_cols:
        z_value = row_lookup(row, f"{col}_robust_abs_z", np.nan)
        if pd.notna(z_value):
            scores.append(float(z_value))
            if float(z_value) >= 3.0:
                reasons.append(f"{col}_robust_outlier")
    if reasons:
        record = row.to_dict()
        record["unusual_score"] = max(scores) if scores else np.nan
        record["unusual_reasons"] = " | ".join(sorted(set(reasons)))
        unusual_records.append(record)

if unusual_records:
    phase6_unusual_image_review_panel = pd.DataFrame(unusual_records).sort_values("unusual_score", ascending=False)
else:
    phase6_unusual_image_review_panel = pd.DataFrame(columns=["panel_archive_path", "unusual_score", "unusual_reasons"])

write_csv(phase6_unusual_image_review_panel, OUTPUT_DIR / "phase6_unusual_image_review_panel.csv")

if not phase6_unusual_image_review_panel.empty:
    for _, row in phase6_unusual_image_review_panel.head(PANEL_MAX_IMAGES).iterrows():
        record_panel_item(
            panel_type="unusual_image_review_panel",
            row=row,
            reason=row_lookup(row, "unusual_reasons", ""),
            panel_field="unusual_reasons",
            panel_value=row_lookup(row, "unusual_reasons", ""),
        )
    make_image_grid_from_rows(
        phase6_unusual_image_review_panel,
        "phase6_20_unusual_image_review_panel.png",
        "Unusual-image Review Panel",
        f"{min(len(phase6_unusual_image_review_panel), PANEL_MAX_IMAGES)} linked readable images with screening flags or robust feature outliers",
        "Review panel ranked by simple flags and robust outlier scores from thumbnail descriptors",
        "Unusual-image flags are screening cues only; they may reflect imaging conditions rather than hydrogel behaviour.",
        label_columns=["unusual_reasons"],
    )


phase6_visual_review_panel_index = pd.DataFrame(visual_review_panel_records)
if phase6_visual_review_panel_index.empty:
    phase6_visual_review_panel_index = pd.DataFrame(columns=[
        "panel_type", "panel_field", "panel_value", "source_csv_record", "image_id", "archive_path", "reason_for_review", "scientific_boundary"
    ])
write_csv(phase6_visual_review_panel_index, OUTPUT_DIR / "phase6_visual_review_panel_index.csv")

figure_register_df = pd.DataFrame(figure_register)
write_csv(figure_register_df, OUTPUT_DIR / "phase6_figure_register.csv")

print("Second-round review panel records:", len(phase6_visual_review_panel_index))
print("Second-round figures now registered:", len(figure_register_df))
display_frame("Second-round Visual Review Panel Index", phase6_visual_review_panel_index, n=20)
display_frame("Updated Phase 6 Figure Register", figure_register_df, n=60)


Second-round review panel records: 88
Second-round figures now registered: 20


### Second-round Visual Review Panel Index

,panel_type,panel_field,panel_value,source_csv_record,image_id,archive_path,reason_for_review,scientific_boundary
0,category_example_grid,ink_type_ph,HA1.0,88,7733,images/resized_IMG_7733.png,Representative thumbnail selected from metadata group ink_type_ph=HA1.0,"Review panel only; visual appearance is not a validated printability, viability or clinical-performance label."
1,category_example_grid,ink_type_ph,HA1.0,89,7734,images/resized_IMG_7734.png,Representative thumbnail selected from metadata group ink_type_ph=HA1.0,"Review panel only; visual appearance is not a validated printability, viability or clinical-performance label."
2,category_example_grid,ink_type_ph,HA1.0,90,7735,images/resized_IMG_7735.png,Representative thumbnail selected from metadata group ink_type_ph=HA1.0,"Review panel only; visual appearance is not a validated printability, viability or clinical-performance label."
3,category_example_grid,ink_type_ph,ALG2.0HA1.0,429,8190,images/resized_IMG_8190.png,Representative thumbnail selected from metadata group ink_type_ph=ALG2.0HA1.0,"Review panel only; visual appearance is not a validated printability, viability or clinical-performance label."
4,category_example_grid,ink_type_ph,ALG2.0HA1.0,430,8191,images/resized_IMG_8191.png,Representative thumbnail selected from metadata group ink_type_ph=ALG2.0HA1.0,"Review panel only; visual appearance is not a validated printability, viability or clinical-performance label."
5,category_example_grid,ink_type_ph,ALG2.0HA1.0,431,8192,images/resized_IMG_8192.png,Representative thumbnail selected from metadata group ink_type_ph=ALG2.0HA1.0,"Review panel only; visual appearance is not a validated printability, viability or clinical-performance label."
6,category_example_grid,ink_type_ph,ALG1.0,10,7595,images/resized_IMG_7595.png,Representative thumbnail selected from metadata group ink_type_ph=ALG1.0,"Review panel only; visual appearance is not a validated printability, viability or clinical-performance label."
7,category_example_grid,ink_type_ph,ALG1.0,11,7596,images/resized_IMG_7596.png,Representative thumbnail selected from metadata group ink_type_ph=ALG1.0,"Review panel only; visual appearance is not a validated printability, viability or clinical-performance label."
8,category_example_grid,ink_type_ph,ALG1.0,12,7597,images/resized_IMG_7597.png,Representative thumbnail selected from metadata group ink_type_ph=ALG1.0,"Review panel only; visual appearance is not a validated printability, viability or clinical-performance label."
9,category_example_grid,ink_type_ph,ALG1.0HA1.0,130,7775,images/resized_IMG_7775.png,Representative thumbnail selected from metadata group ink_type_ph=ALG1.0HA1.0,"Review panel only; visual appearance is not a validated printability, viability or clinical-performance label."


### Updated Phase 6 Figure Register

,figure_file,title,records_analysed,method,scientific_interpretation,limitations,relative_path
0,phase6_01_image_file_format_counts.png,Image archive file-format counts,1031 image candidate files,ZIP inventory grouped by file suffix,Image archive file-format counts,"File suffix is a storage format cue, not a scientific measurement.",data\processed\image_file_audit_optional\figures\phase6_01_image_file_format_counts.png
1,phase6_02_image_index_matching_coverage.png,Image-index to archive matching coverage,1031 image-index records,Metadata-to-file matching using documented filename rules,Image-index to archive matching coverage,File matching does not prove physical-sample identity.,data\processed\image_file_audit_optional\figures\phase6_02_image_index_matching_coverage.png
2,phase6_03_image_dimensions_scatter.png,Readable image pixel dimensions,1031 readable images,Pillow image metadata extraction,Readable image pixel dimensions,"Pixel dimensions describe digital files, not physical print dimensions unless calibration metadata is available.",data\processed\image_file_audit_optional\figures\phase6_03_image_dimensions_scatter.png
3,phase6_04_metadata_missingness_map.png,Image metadata missingness map,1031 image-index records,Binary missingness display across selected metadata fields,Image metadata missingness map,Missingness records source availability only; missing values are not imputed.,data\processed\image_file_audit_optional\figures\phase6_04_metadata_missingness_map.png
4,phase6_05_ink_type_distribution.png,Ink-type Metadata Distribution,1031 image-index records,Value counts for image_index.ink_type_ph,Ink-type Metadata Distribution,Metadata categories describe source labels; they are not automatically standardised experimental variables.,data\processed\image_file_audit_optional\figures\phase6_05_ink_type_distribution.png
5,phase6_06_printed_shape_distribution.png,Printed-shape Metadata Distribution,1031 image-index records,Value counts for image_index.shape,Printed-shape Metadata Distribution,Metadata categories describe source labels; they are not automatically standardised experimental variables.,data\processed\image_file_audit_optional\figures\phase6_06_printed_shape_distribution.png
6,phase6_07_irradiation_intensity_distribution.png,Irradiation-intensity Metadata Distribution,1031 image-index records,Value counts for image_index.intensity,Irradiation-intensity Metadata Distribution,Metadata categories describe source labels; they are not automatically standardised experimental variables.,data\processed\image_file_audit_optional\figures\phase6_07_irradiation_intensity_distribution.png
7,phase6_08_brightness_contrast_scatter.png,Image brightness and contrast screening,1031 readable images,Thumbnail grayscale intensity summary,Image brightness and contrast screening,Brightness and contrast may reflect imaging conditions as well as printed material appearance.,data\processed\image_file_audit_optional\figures\phase6_08_brightness_contrast_scatter.png
8,phase6_09_foreground_area_by_shape.png,Foreground-area fraction by recorded shape category,1031 linked readable images,Border-background thresholding on thumbnail images,Foreground-area fraction by recorded shape category,Foreground area is a computed descriptor and not a validated printability score.,data\processed\image_file_audit_optional\figures\phase6_09_foreground_area_by_shape.png
9,phase6_10_example_image_grid.png,Example source image grid,12 readable source images,First readable images loaded from the ZIP archive and displayed as thumbnails,Example source image grid,The grid supports visual audit only and is not a representative statistical sample unless sampling is defined.,data\processed\image_file_audit_optional\figures\phase6_10_example_image_grid.png


## Quality Control and Scientific Boundary Checks

A good data-curation notebook should separate programming failures from scientific review findings. For example, an unreadable ZIP archive is a blocking technical failure. But a missing metadata value, an unmatched image, or an unusual brightness distribution is usually a review item, not a reason to silently delete the record.

This is especially important in materials informatics, where small datasets and imperfect source metadata are common. The correct response is traceable uncertainty, not invented certainty.

In [28]:
# QC findings and validation summary.

qc_records = []

def add_qc(finding_type, severity, source_table, evidence, action, interpretation):
    qc_records.append({
        "finding_id": f"P6_QC_{len(qc_records) + 1:04d}",
        "finding_type": finding_type,
        "severity": severity,
        "source_table": source_table,
        "evidence": evidence,
        "recommended_action": action,
        "phase6_interpretation": interpretation,
    })

not_matched_count = int((image_index_to_archive_linkage["linkage_status"] == "not_matched_review").sum())
if not_matched_count:
    add_qc(
        "image_index_records_not_matched_to_archive",
        "review",
        "phase6_image_index_to_archive_linkage.csv",
        f"{not_matched_count} image-index records were not matched to an archive image file.",
        "Review file names, source archive structure and image_id conventions.",
        "Unmatched metadata should remain unresolved until a source-supported mapping is found.",
    )

multi_match_count = int((image_index_to_archive_linkage["linkage_status"] == "matched_multiple_files_review").sum())
if multi_match_count:
    add_qc(
        "image_index_records_matched_multiple_files",
        "review",
        "phase6_image_index_to_archive_linkage.csv",
        f"{multi_match_count} image-index records matched multiple archive files.",
        "Review duplicate names or repeated export variants before selecting a primary file.",
        "Multiple file matches do not justify automatic record collapse.",
    )

archive_only_count = len(archive_only_images)
if archive_only_count:
    add_qc(
        "archive_images_not_represented_in_image_index",
        "review",
        "phase6_archive_only_images.csv",
        f"{archive_only_count} archive image files were not linked to image_index records.",
        "Review whether these are extras, processed outputs, thumbnails, or naming mismatches.",
        "Archive-only images remain source evidence but should not inherit metadata labels without support.",
    )

unreadable_count = int((image_integrity_properties["readable"] == False).sum())
if unreadable_count:
    add_qc(
        "unreadable_image_files",
        "warning",
        "phase6_image_integrity_properties.csv",
        f"{unreadable_count} image files could not be opened by Pillow.",
        "Inspect corrupted files or unsupported formats.",
        "Unreadable images cannot support visual feature extraction until resolved.",
    )

if not duplicate_image_diagnostics.empty:
    add_qc(
        "duplicate_or_near_duplicate_image_diagnostics",
        "review",
        "phase6_duplicate_image_diagnostics.csv",
        f"{len(duplicate_image_diagnostics)} duplicate diagnostic groups were detected.",
        "Review exact and thumbnail-hash duplicate groups before any modelling split.",
        "Duplicates can cause data leakage if the same visual evidence appears in both train and test sets.",
    )

for row in metadata_completeness.itertuples(index=False):
    if getattr(row, "missing") > 0:
        add_qc(
            "metadata_missing_values",
            "review",
            "phase6_metadata_completeness.csv",
            f"Field {getattr(row, 'field')} has {getattr(row, 'missing')} missing values.",
            "Retain missingness and avoid inventing labels or concentrations.",
            "Missing metadata is part of the evidence layer and should not be imputed as source fact.",
        )

if "blank_like_flag" in image_integrity_properties.columns:
    blank_count = int(image_integrity_properties["blank_like_flag"].fillna(False).sum())
    if blank_count:
        add_qc(
            "blank_like_image_screening_flag",
            "review",
            "phase6_image_integrity_properties.csv",
            f"{blank_count} readable images had very low grayscale variation.",
            "Visually review before interpreting as blank, failed or low-information images.",
            "A blank-like image-processing flag is not automatically an experimental failure label.",
        )

phase6_qc_findings = pd.DataFrame(qc_records)
write_csv(phase6_qc_findings, OUTPUT_DIR / "phase6_qc_findings.csv")

validation_records = []

def add_validation(check, status, severity, evidence):
    validation_records.append({
        "check": check,
        "status": status,
        "severity": severity,
        "evidence": evidence,
    })

add_validation("images_zip_exists", "passed" if IMAGE_ZIP_PATH.exists() else "failed", "critical", str(IMAGE_ZIP_PATH))
add_validation("image_index_loaded", "passed" if len(image_index) > 0 else "failed", "critical", f"{len(image_index)} records")
add_validation("zip_integrity_test", "passed" if zip_integrity_bad_member is None else "failed", "critical", f"bad_member={zip_integrity_bad_member}")
add_validation("archive_has_image_files", "passed" if len(image_file_inventory) > 0 else "failed", "critical", f"{len(image_file_inventory)} image files")
add_validation("image_files_readable", "passed" if unreadable_count == 0 else "warning", "warning", f"{unreadable_count} unreadable image files")
add_validation("image_index_archive_matching_complete", "passed" if not_matched_count == 0 and multi_match_count == 0 else "review", "review", f"{not_matched_count} unmatched; {multi_match_count} multiple matches")
add_validation("archive_only_images", "passed" if archive_only_count == 0 else "review", "review", f"{archive_only_count} archive-only image files")
add_validation("phase4_supporting_context_available", "passed" if PHASE4_DIR.exists() else "warning", "warning", str(PHASE4_DIR))
add_validation("phase5_supporting_context_available", "passed" if PHASE5_DIR.exists() else "warning", "warning", str(PHASE5_DIR))
add_validation("scientific_boundary_preserved", "passed", "critical", "Notebook does not infer physical-sample identity, printability success, cell viability, tissue performance, or clinical outcomes from image features.")

phase6_validation_summary = pd.DataFrame(validation_records)
write_csv(phase6_validation_summary, OUTPUT_DIR / "phase6_validation_summary.csv")

display_frame("Phase 6 QC Findings", phase6_qc_findings, n=30)
display_frame("Phase 6 Validation Summary", phase6_validation_summary, n=30)

### Phase 6 QC Findings

,finding_id,finding_type,severity,source_table,evidence,recommended_action,phase6_interpretation
0,P6_QC_0001,duplicate_or_near_duplicate_image_diagnostics,review,phase6_duplicate_image_diagnostics.csv,2 duplicate diagnostic groups were detected.,Review exact and thumbnail-hash duplicate groups before any modelling split.,Duplicates can cause data leakage if the same visual evidence appears in both train and test sets.
1,P6_QC_0002,metadata_missing_values,review,phase6_metadata_completeness.csv,Field completeness_sam has 943 missing values.,Retain missingness and avoid inventing labels or concentrations.,Missing metadata is part of the evidence layer and should not be imputed as source fact.
2,P6_QC_0003,metadata_missing_values,review,phase6_metadata_completeness.csv,Field contain_sps has 922 missing values.,Retain missingness and avoid inventing labels or concentrations.,Missing metadata is part of the evidence layer and should not be imputed as source fact.


### Phase 6 Validation Summary

,check,status,severity,evidence
0,images_zip_exists,passed,critical,C:\Users\sufer\Documents\hydrogel-bioink\data\raw\zenodo_19602891\images.zip
1,image_index_loaded,passed,critical,1031 records
2,zip_integrity_test,passed,critical,bad_member=None
3,archive_has_image_files,passed,critical,1031 image files
4,image_files_readable,passed,warning,0 unreadable image files
5,image_index_archive_matching_complete,passed,review,0 unmatched; 0 multiple matches
6,archive_only_images,passed,review,0 archive-only image files
7,phase4_supporting_context_available,passed,warning,C:\Users\sufer\Documents\hydrogel-bioink\data\processed\record_linkage_and_analysis_ready_evidence_layer
8,phase5_supporting_context_available,passed,warning,C:\Users\sufer\Documents\hydrogel-bioink\data\processed\hydrogel_rheology_and_printing_context_analysis
9,scientific_boundary_preserved,passed,critical,"Notebook does not infer physical-sample identity, printability success, cell viability, tissue performance, or clinical outcomes from image features."


## Output Data Dictionary, Gallery and Run Manifest

The final section records what the notebook produced. This supports GitHub presentation and future reproducibility.

The output files separate:

- source archive inventory;
- source metadata profiling;
- metadata-to-file linkage;
- image integrity and computed descriptors;
- QC findings;
- validation checks;
- visual figure register;
- run manifest.

That separation is important because original observations, source metadata, processed images, and computed features should not be mixed as if they were the same type of evidence.

In [30]:
# Data dictionary, HTML figure gallery, output inventory and run manifest.

outputs = [
    ("phase6_input_verification.csv", "CSV", "Input file availability, size and checksum where appropriate."),
    ("phase6_image_index_profile.csv", "CSV", "Field-level profile of image_index.csv."),
    ("phase6_duplicate_image_ids.csv", "CSV", "Duplicate image_id rows from image_index.csv, if present."),
    ("phase6_zip_file_inventory.csv", "CSV", "All ZIP archive members and basic ZIP metadata."),
    ("phase6_image_file_inventory.csv", "CSV", "Archive members that look like image files."),
    ("phase6_image_index_to_archive_linkage.csv", "CSV", "Traceable linkage between image_index records and archive image files."),
    ("phase6_archive_only_images.csv", "CSV", "Image files present in the archive but not linked to image_index records."),
    ("phase6_image_integrity_properties.csv", "CSV", "Image readability, dimensions, modes and computed descriptors."),
    ("phase6_image_feature_summary.csv", "CSV", "Compact table of computed exploratory image features."),
    ("phase6_duplicate_image_diagnostics.csv", "CSV", "Exact SHA-256 and thumbnail-hash duplicate review cues."),
    ("phase6_image_metadata_feature_layer.csv", "CSV", "Metadata linked to primary archive files and computed features where available."),
    ("phase6_metadata_completeness.csv", "CSV", "Metadata completeness and missingness interpretation."),
    ("phase6_metadata_group_feature_summary.csv", "CSV", "Descriptive feature summaries by selected metadata fields."),
    ("phase6_visual_group_feature_summary.csv", "CSV", "Second-round median image descriptors by selected metadata groups for visual review."),
    ("phase6_visual_review_panel_index.csv", "CSV", "Index of images included in second-round visual review panels and the reason each image was selected."),
    ("phase6_missing_metadata_review_panel.csv", "CSV", "Rows with missing selected image metadata fields, retained for source review."),
    ("phase6_duplicate_group_review_panel.csv", "CSV", "Images selected from duplicate or near-duplicate diagnostic groups for manual review."),
    ("phase6_unusual_image_review_panel.csv", "CSV", "Images selected by simple flags or robust thumbnail-feature outlier scores for manual review."),
    ("phase6_qc_findings.csv", "CSV", "Traceable QC findings and review items."),
    ("phase6_validation_summary.csv", "CSV", "Validation checks with passed, warning, review or failed statuses."),
    ("phase6_figure_register.csv", "CSV", "Figure files, methods, records analysed and interpretation limits."),
    ("phase6_data_dictionary.csv", "CSV", "Descriptions of Phase 6 output files."),
    ("phase6_run_manifest.json", "JSON", "Run timestamp, environment, input paths, counts and output inventory."),
    ("phase6_figure_gallery.html", "HTML", "Browser-viewable Phase 6 figure gallery."),
]

phase6_data_dictionary = pd.DataFrame(outputs, columns=["output_file", "output_type", "description"])
write_csv(phase6_data_dictionary, OUTPUT_DIR / "phase6_data_dictionary.csv")

# Build figure gallery.
gallery_path = OUTPUT_DIR / "phase6_figure_gallery.html"
figure_cards = []
for row in figure_register_df.itertuples(index=False):
    rel_to_gallery = Path(CONFIG["figures_subdir_name"]) / getattr(row, "figure_file")
    figure_cards.append(f"""
    <section class="figure-card">
      <h2>{html.escape(getattr(row, 'title'))}</h2>
      <img src="{html.escape(str(rel_to_gallery).replace(os.sep, '/'))}" alt="{html.escape(getattr(row, 'title'))}">
      <p><strong>Records analysed:</strong> {html.escape(str(getattr(row, 'records_analysed')))}</p>
      <p><strong>Method:</strong> {html.escape(str(getattr(row, 'method')))}</p>
      <p><strong>Limitations:</strong> {html.escape(str(getattr(row, 'limitations')))}</p>
    </section>
    """)

gallery_html = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>Phase 6 Figure Gallery</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 32px; color: #222; line-height: 1.45; }}
    h1 {{ margin-bottom: 0.2rem; }}
    .intro {{ max-width: 980px; }}
    .figure-card {{ border-top: 1px solid #ddd; padding: 24px 0; max-width: 1100px; }}
    img {{ max-width: 100%; height: auto; border: 1px solid #ddd; }}
    p {{ max-width: 980px; }}
  </style>
</head>
<body>
  <h1>Phase 6 Figure Gallery</h1>
  <p class="intro">Hydrogel bioink image-file audit, image metadata validation and exploratory image analysis. Figures include archive audit charts, metadata-linkage charts and second-round visual review panels. They are descriptive and must not be interpreted as validated printability, cell viability, tissue-performance or clinical-outcome measures.</p>
  {''.join(figure_cards)}
</body>
</html>
"""
with open(gallery_path, "w", encoding="utf-8") as f:
    f.write(gallery_html)

# Output inventory after all writes.
output_inventory_records = []
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        output_inventory_records.append({
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "file_name": path.name,
            "file_size_bytes": int(path.stat().st_size),
            "suffix": path.suffix.lower(),
        })
phase6_output_inventory = pd.DataFrame(output_inventory_records)
write_csv(phase6_output_inventory, OUTPUT_DIR / "phase6_output_inventory.csv")

run_finished_at = dt.datetime.now(dt.timezone.utc)
manifest = {
    "run_id": RUN_ID,
    "phase": CONFIG["phase_name"],
    "started_at_utc": RUN_STARTED_AT.isoformat(),
    "finished_at_utc": run_finished_at.isoformat(),
    "duration_seconds": (run_finished_at - RUN_STARTED_AT).total_seconds(),
    "project_root": str(PROJECT_ROOT),
    "raw_dir": str(RAW_DIR),
    "output_dir": str(OUTPUT_DIR),
    "python_executable": sys.executable,
    "python_version": sys.version,
    "platform": platform.platform(),
    "scikit_image_requested": SKIMAGE_REQUESTED,
    "scikit_image_available": SKIMAGE_AVAILABLE,
    "scikit_image_import_status": SKIMAGE_IMPORT_STATUS,
    "scikit_image_import_message": SKIMAGE_IMPORT_MESSAGE,
    "threshold_fallback": "numpy_histogram_otsu_when_scikit_image_unavailable",
    "config": CONFIG,
    "input_summary": input_verification.to_dict(orient="records"),
    "key_counts": {
        "image_index_records": int(len(image_index)),
        "zip_members": int(len(zip_inventory)),
        "image_candidate_files": int(len(image_file_inventory)),
        "readable_images": int(image_integrity_properties["readable"].sum()),
        "unreadable_images": int((image_integrity_properties["readable"] == False).sum()),
        "matched_one_file_records": int((image_index_to_archive_linkage["linkage_status"] == "matched_one_file").sum()),
        "not_matched_records": int((image_index_to_archive_linkage["linkage_status"] == "not_matched_review").sum()),
        "archive_only_images": int(len(archive_only_images)),
        "qc_findings": int(len(phase6_qc_findings)),
        "figures": int(len(figure_register_df)),
        "second_round_visual_review_records": int(len(phase6_visual_review_panel_index)) if "phase6_visual_review_panel_index" in globals() else 0,
        "missing_metadata_review_records": int(len(phase6_missing_metadata_review_panel)) if "phase6_missing_metadata_review_panel" in globals() else 0,
        "duplicate_group_review_records": int(len(phase6_duplicate_group_review_panel)) if "phase6_duplicate_group_review_panel" in globals() else 0,
        "unusual_image_review_records": int(len(phase6_unusual_image_review_panel)) if "phase6_unusual_image_review_panel" in globals() else 0,
    },
    "scientific_boundaries": [
        "Image features are computed descriptors, not validated printability outcomes.",
        "File-name matches do not prove physical-sample identity.",
        "Missing metadata is preserved and not invented.",
        "No inference is made about cell viability, tissue performance or clinical outcomes.",
        "Original experimental images, metadata, processed descriptors and AI-generated outputs must remain separated.",
    ],
    "output_inventory": phase6_output_inventory.to_dict(orient="records"),
}
write_json(manifest, OUTPUT_DIR / "phase6_run_manifest.json")

print("Phase 6 outputs written to:", OUTPUT_DIR)
print("Figure gallery:", gallery_path)
display_frame("Phase 6 Output Inventory", phase6_output_inventory, n=50)

Phase 6 outputs written to: C:\Users\sufer\Documents\hydrogel-bioink\data\processed\image_file_audit_optional
Figure gallery: C:\Users\sufer\Documents\hydrogel-bioink\data\processed\image_file_audit_optional\phase6_figure_gallery.html


### Phase 6 Output Inventory

,relative_path,file_name,file_size_bytes,suffix
0,data\processed\image_file_audit_optional\figures\phase6_01_image_file_format_counts.png,phase6_01_image_file_format_counts.png,27444,.png
1,data\processed\image_file_audit_optional\figures\phase6_02_image_index_matching_coverage.png,phase6_02_image_index_matching_coverage.png,33705,.png
2,data\processed\image_file_audit_optional\figures\phase6_03_image_dimensions_scatter.png,phase6_03_image_dimensions_scatter.png,37013,.png
3,data\processed\image_file_audit_optional\figures\phase6_04_metadata_missingness_map.png,phase6_04_metadata_missingness_map.png,41264,.png
4,data\processed\image_file_audit_optional\figures\phase6_05_ink_type_distribution.png,phase6_05_ink_type_distribution.png,73911,.png
5,data\processed\image_file_audit_optional\figures\phase6_06_printed_shape_distribution.png,phase6_06_printed_shape_distribution.png,28000,.png
6,data\processed\image_file_audit_optional\figures\phase6_07_irradiation_intensity_distribution.png,phase6_07_irradiation_intensity_distribution.png,27279,.png
7,data\processed\image_file_audit_optional\figures\phase6_08_brightness_contrast_scatter.png,phase6_08_brightness_contrast_scatter.png,119493,.png
8,data\processed\image_file_audit_optional\figures\phase6_09_foreground_area_by_shape.png,phase6_09_foreground_area_by_shape.png,45358,.png
9,data\processed\image_file_audit_optional\figures\phase6_10_example_image_grid.png,phase6_10_example_image_grid.png,2510944,.png


## Phase 6 Interpretation Guide

This notebook creates a defensible image evidence layer for the hydrogel bioink project.

What Phase 6 can support:

- whether the original image archive is readable;
- how many image files are present;
- whether image-index metadata can be linked to actual files;
- whether metadata fields are complete or missing;
- basic digital image properties such as dimensions, colour mode, brightness, contrast, edge density and foreground-area fraction;
- visual review panels by ink type, shape and irradiation intensity;
- review panels for missing metadata, duplicate cues and unusual images;
- traceable review items for ambiguous or unusual evidence.

Interpreting the richer visualisations:

- **Grouped image grids** provide a visual check of whether metadata labels such as ink type, shape or intensity are linked to plausible source images.
- **Feature heatmaps** show whether simple image descriptors vary across recorded metadata groups. They are scaled for visual comparison, not for physical measurement.
- **Missing-metadata panels** identify source records where metadata requires review instead of assumption-based completion.
- **Duplicate panels** reduce future modelling leakage risk by flagging cases where the same or near-same image evidence may appear in more than one record.
- **Unusual-image panels** flag very bright, dark, low-contrast or feature-outlying images for review before interpretation.

What Phase 6 cannot prove by itself:

- that a printed construct is biologically successful;
- that one formulation is clinically better than another;
- that a rheology measurement and an image came from the same physical sample;
- that a computed image feature is a validated printability metric;
- that a bright, dark, high-foreground or low-contrast image is automatically a successful or failed hydrogel print;
- that missing concentrations, dates or labels can be filled in from assumptions.

The important idea is simple: data science organises and screens evidence; polymer science defines the physical meaning that the evidence may or may not carry. That conservative boundary is not a weakness. It is what makes the project credible.
